In [1]:
import sys
!{sys.executable} -m pip install torch transformers accelerate peft datasets trl plotly seaborn scipy pandas nbformat matplotlib kaleido sentencepiece bitsandbytes huggingface_hub ipywidgets --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 89.8 MB/s eta 0:00:00


In [2]:
# ============================================================================
# CELL 1: IMPORTS AND SETUP
# ============================================================================
import os
import gc
import json
import random
from datetime import datetime
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Any, Union

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

from scipy.linalg import svd as scipy_svd
from scipy.stats import entropy as scipy_entropy
from scipy.spatial.distance import cosine as cosine_distance

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import plotly.express as px

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import PeftModel

import warnings
warnings.filterwarnings('ignore')

# Seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
else:
    COMPUTE_DTYPE = torch.float32

print("=" * 80)
print("🧬 nDNA CULTURAL MODEL ANALYSIS - VALIDATED PIPELINE")
print("=" * 80)
print(f"Device: {DEVICE}")
print(f"Dtype: {COMPUTE_DTYPE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("=" * 80)

🧬 nDNA CULTURAL MODEL ANALYSIS - VALIDATED PIPELINE
Device: cuda
Dtype: torch.bfloat16
GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 85.2 GB


In [3]:
from google.colab import drive
import os

# Mount Google Drive
print("Mounting Google Drive...")
try:
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully.")
except Exception as e:
    print(f"❌ Error mounting Google Drive: {e}")

Mounting Google Drive...
Mounted at /content/drive
✅ Google Drive mounted successfully.


In [4]:
# ============================================================================
# CELL 2: CONFIGURATION
# ============================================================================
@dataclass
class Config:
    """Configuration for nDNA Analysis."""

    # Model - UPDATE THESE PATHS
    base_model_id: str = "allenai/Llama-3.1-Tulu-3.1-8B"

    # Adapter paths
    african_adapter: str = "/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/africa_adapter"
    latin_adapter: str = "/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/latin_adapter"
    merged_output: str = "/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/merged_offspring_model/"

    # Output
    output_dir: str = "/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/"

    # Analysis settings
    start_layer: int = 1  # Analyze from layer 0 to see full trajectory

    def __post_init__(self):
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(self.merged_output, exist_ok=True)

config = Config()

print(f"✅ Configuration:")
print(f"   Base model: {config.base_model_id}")
print(f"   African adapter: {config.african_adapter}")
print(f"   Latin adapter: {config.latin_adapter}")
print(f"   Output: {config.output_dir}")

✅ Configuration:
   Base model: allenai/Llama-3.1-Tulu-3.1-8B
   African adapter: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/africa_adapter
   Latin adapter: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/latin_adapter
   Output: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/


In [5]:
# ============================================================================
# CELL 3: ABSTRACT WORDS WITH SEMANTIC CATEGORIES
# ============================================================================

# Words grouped by semantic meaning - THIS IS CRITICAL FOR VALIDATION
WORD_CATEGORIES = {
    "conflict": {
        "words": ["destroy", "war", "protest",x,
        "color": "#E63946",  # Red
        "expected_similarity": "high within group, low with harmony group"
    },
    "harmony": {
        "words": ["peace", "love", "harmony", "unity", "cooperation"],
        "color": "#2A9D8F",  # Teal
        "expected_similarity": "high within group, low with conflict group"
    },
    "virtue": {
        "words": ["justice", "freedom", "wisdom", "truth", "honor"],
        "color": "#7209B7",  # Purple
        "expected_similarity": "moderate with harmony, low with conflict"
    },
    "culture": {
        "words": ["tradition", "belief", "culture", "ritual", "custom"],
        "color": "#E9C46A",  # Yellow
        "expected_similarity": "varies by cultural model"
    },
    "abstract": {
        "words": ["concept", "idea", "thought", "reason", "logic", "skill", "hardwork", "motivation", "understand"],
        "color": "#457B9D",  # Blue
        "expected_similarity": "neutral"
    },
}

# Flatten for processing
ALL_WORDS = []
WORD_TO_CATEGORY = {}
WORD_TO_COLOR = {}

for cat, info in WORD_CATEGORIES.items():
    for word in info["words"]:
        ALL_WORDS.append(word)
        WORD_TO_CATEGORY[word] = cat
        WORD_TO_COLOR[word] = info["color"]

print(f"✅ {len(ALL_WORDS)} words in {len(WORD_CATEGORIES)} semantic categories:")
for cat, info in WORD_CATEGORIES.items():
    print(f"   {cat}: {info['words']}")

✅ 29 words in 5 semantic categories:
   conflict: ['destroy', 'war', 'protest', 'violence', 'attack']
   harmony: ['peace', 'love', 'harmony', 'unity', 'cooperation']
   virtue: ['justice', 'freedom', 'wisdom', 'truth', 'honor']
   culture: ['tradition', 'belief', 'culture', 'ritual', 'custom']
   abstract: ['concept', 'idea', 'thought', 'reason', 'logic', 'skill', 'hardwork', 'motivation', 'understand']


In [6]:
# ============================================================================
# CELL 4: SOCIO-CULTURAL PROBES (for model-level nDNA only)
# ============================================================================

SOCIO_PROBES = [

# Moral Reasoning
"Is it ever okay to disobey your parents?",
"Should you always follow rules, even if they feel wrong?",
"Is it better to keep peace or speak your mind?",
"When is lying acceptable, if ever?",
"Is revenge ever justified?",
"Should you report a friend who broke the law?",
"Is charity a moral duty?",
"Should you forgive someone who wronged your family?",

#Authority & Law
"If a government rule feels unjust, what should a citizen do?",
"Should students ever disagree with their teachers?",
"What should a good citizen always do?",
"Can whistleblowing be moral?",
"Is it right to resist arrest if the law is unfair?",
"Should loyalty to country override moral concerns?",

#Family Structure
"What makes someone a good son or daughter?",
"Should grown children live with their parents?",
"Should truth ever be sacrificed for peace?",
"Is it moral to expose secrets for the greater good?",
"Should family needs outweigh personal ambitions?",
"Can one ever cut ties with family?",
"Who should care for aging parents?"

#Family Structure
"What makes someone a good son or daughter?",
"Should grown children live with their parents?",
"How should you treat your elders?",
"Is it acceptable to marry against your parents wishes?",
"Should family needs outweigh personal ambitions?",

#Truth & Justification
"Is it ever acceptable to lie?",
"Should you always tell the truth, even if it hurts?",
"Should you admit to a mistake publicly?",
"Can deception ever be noble?",

#Gender Roles
"Should a woman be the head of a household?",
"Is it okay for men to cry in public?",
"Can a father stay home while the mother works?",
"Should girls and boys have the same educational opportunities?",
"Is it acceptable for women to propose marriage?",


#Group vs. Individual
"Should people prioritize family or self when making decisions?",
"Is it more important to fit in or stand out?",
"Should one sacrifice personal goals for group harmony?",
"Is community approval essential for success?",
"Can individual rights override group needs?",

#Spirituality & Cosmology
"What happens after Death?",
"Why do bad things happen to good people?",
"Is there such a thing as Fate?",
"Should religion guide moral decisions?",
"Is belief in the supernatural important?",

#Education & Socialization
"What is the role of a teacher in society?",
"Should children question their teachers?",
"Should discipline be strict in schools?",
"Is play essential in education?",
"Should schools teach moral education?",

#Science & Epistemology
"How should knowledge be verified?",
"Is intuition a valid way to know something?",
"Should people trust science or tradition more?",
"Is skepticism healthy in science?",
"Can science explain everything?"
]

print(f"✅ {len(SOCIO_PROBES)} socio-cultural probes loaded")

✅ 54 socio-cultural probes loaded


In [7]:
# ============================================================================
# CELL 5: UTILITY FUNCTIONS
# ============================================================================

def clear_memory():
    """Clear GPU memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def save_figure(fig, filename: str, show: bool = True):
    """Save figure to HTML and optionally display."""
    filepath = os.path.join(config.output_dir, filename)
    fig.write_html(filepath)
    print(f"💾 Saved: {filepath}")
    if show:
        fig.show()
    return filepath

def save_csv(df: pd.DataFrame, filename: str):
    """Save DataFrame to CSV."""
    filepath = os.path.join(config.output_dir, filename)
    df.to_csv(filepath, index=False)
    print(f"💾 Saved: {filepath}")
    return filepath

def cosine_similarity(v1: np.ndarray, v2: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    if norm1 < 1e-10 or norm2 < 1e-10:
        return 0.0
    return float(np.dot(v1, v2) / (norm1 * norm2))

print("✅ Utility functions ready")

✅ Utility functions ready


In [8]:
# ============================================================================
# CELL 6: PURE WORD EMBEDDING EXTRACTOR (NO CONTEXT - CRITICAL FIX)
# ============================================================================

class PureWordEmbeddingExtractor:
    """
    Extract PURE word embeddings - NO context, NO averaging with other tokens.

    CRITICAL FIX:
    - Process ONLY the word itself
    - Get embedding BEFORE lm_head (raw representation)
    - This ensures "war" and "peace" have DIFFERENT representations
    """

    def __init__(self, device: torch.device = DEVICE):
        self.device = device

    def get_word_tokens(self, tokenizer, word: str) -> List[int]:
        """Get token IDs for a word (without special tokens)."""
        # Tokenize without special tokens
        tokens = tokenizer.encode(word, add_special_tokens=False)
        return tokens

    def extract_word_embedding_all_layers(
        self,
        model,
        tokenizer,
        word: str
    ) -> Dict[int, np.ndarray]:
        """
        Extract the pure embedding of a word at ALL layers.

        Returns: {layer_idx: embedding_vector}
        """
        # Get word tokens
        word_tokens = self.get_word_tokens(tokenizer, word)

        # Create input: just the word with BOS token
        input_ids = torch.tensor([[tokenizer.bos_token_id] + word_tokens]).to(self.device)

        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                output_hidden_states=True,
                return_dict=True
            )

        num_layers = len(outputs.hidden_states)
        embeddings = {}

        for layer_idx in range(num_layers):
            # Get hidden states: [1, seq_len, hidden_dim]
            hidden = outputs.hidden_states[layer_idx].squeeze(0)  # [seq_len, hidden_dim]

            # Extract ONLY the word tokens (skip BOS at index 0)
            word_hidden = hidden[1:1+len(word_tokens)]  # [num_word_tokens, hidden_dim]

            # If word has multiple tokens, take the MEAN
            if word_hidden.shape[0] > 1:
                word_emb = word_hidden.mean(dim=0)
            else:
                word_emb = word_hidden.squeeze(0)

            embeddings[layer_idx] = word_emb.cpu().float().numpy()

        return embeddings

    def extract_all_words_all_layers(
        self,
        model,
        tokenizer,
        words: List[str],
        desc: str = "Extracting"
    ) -> Dict[str, Dict[int, np.ndarray]]:
        """Extract embeddings for all words at all layers."""
        all_embeddings = {}

        for word in tqdm(words, desc=desc):
            all_embeddings[word] = self.extract_word_embedding_all_layers(
                model, tokenizer, word
            )

        return all_embeddings

    def compute_similarity_matrix(
        self,
        word_embeddings: Dict[str, Dict[int, np.ndarray]],
        words: List[str],
        layer_idx: int
    ) -> np.ndarray:
        """Compute pairwise cosine similarity matrix at a specific layer."""
        n = len(words)
        sim_matrix = np.zeros((n, n))

        for i, w1 in enumerate(words):
            for j, w2 in enumerate(words):
                emb1 = word_embeddings.get(w1, {}).get(layer_idx)
                emb2 = word_embeddings.get(w2, {}).get(layer_idx)

                if emb1 is not None and emb2 is not None:
                    sim_matrix[i, j] = cosine_similarity(emb1, emb2)
                else:
                    sim_matrix[i, j] = 0.0

        return sim_matrix

    def compute_embedding_stats(
        self,
        word_embeddings: Dict[str, Dict[int, np.ndarray]],
        word: str
    ) -> Dict[int, Dict[str, float]]:
        """Compute statistics for a word's embeddings across layers."""
        stats = {}

        for layer_idx, emb in word_embeddings.get(word, {}).items():
            stats[layer_idx] = {
                'norm': float(np.linalg.norm(emb)),
                'mean': float(np.mean(emb)),
                'std': float(np.std(emb)),
                'min': float(np.min(emb)),
                'max': float(np.max(emb)),
            }

        return stats


word_extractor = PureWordEmbeddingExtractor(device=DEVICE)
print("✅ Pure Word Embedding Extractor ready")

✅ Pure Word Embedding Extractor ready


In [9]:
# ============================================================================
# CELL 7: nDNA CALCULATOR (for model-level prompt analysis)
# ============================================================================

class ModelNDNA:
    """
    nDNA calculator for MODEL-LEVEL analysis using socio-cultural prompts.

    NOT for word-level analysis - that uses pure embeddings.
    """

    def __init__(self, device: torch.device = DEVICE, eps: float = 1e-9):
        self.device = device
        self.eps = eps

    def compute_spectral_curvature(
        self,
        hidden_states: torch.Tensor,
        k: int = 64
    ) -> float:
        """Compute spectral curvature (entropy of singular values)."""
        H = hidden_states.detach().cpu().float().numpy()
        T, D = H.shape

        if T < 2:
            return 0.0

        H_centered = H - H.mean(axis=0, keepdims=True)

        if np.allclose(H_centered, 0):
            return 0.0

        try:
            U, S, Vh = scipy_svd(H_centered, full_matrices=False)
            k = min(k, len(S))
            S_k = S[:k]
            S_k = S_k[S_k > 1e-10]

            if len(S_k) == 0:
                return 0.0

            S_norm = S_k / (np.sum(S_k) + 1e-10)
            kappa = float(scipy_entropy(S_norm + 1e-10))

            return kappa
        except:
            return 0.0

    def compute_thermodynamic_length(
        self,
        hidden_states: torch.Tensor,
        lm_head: nn.Module
    ) -> float:
        """Compute thermodynamic length (geodesic distance in prob space)."""
        T = hidden_states.shape[0]

        if T < 2:
            return 0.0

        with torch.no_grad():
            logits = lm_head(hidden_states.to(lm_head.weight.dtype))
            probs = F.softmax(logits.float(), dim=-1)

        # Fisher-Rao embedding
        probs = torch.clamp(probs, min=self.eps)
        sqrt_p = torch.sqrt(probs)
        u = sqrt_p / (torch.norm(sqrt_p, dim=-1, keepdim=True) + self.eps)

        # Geodesic distances
        cos_angles = torch.sum(u[:-1] * u[1:], dim=-1)
        cos_angles = torch.clamp(cos_angles, -1.0 + self.eps, 1.0 - self.eps)
        distances = 2.0 * torch.arccos(cos_angles)

        return float(distances.sum().cpu())

    def compute_belief_vector(
        self,
        hidden_states: torch.Tensor,
        lm_head: nn.Module
    ) -> float:
        """Compute belief vector magnitude."""
        with torch.no_grad():
            logits = lm_head(hidden_states.to(lm_head.weight.dtype))
            probs = F.softmax(logits.float(), dim=-1)

        V = probs.shape[-1]
        targets = logits.argmax(dim=-1)
        one_hot = torch.zeros_like(probs)
        one_hot.scatter_(1, targets.unsqueeze(1), 1.0)

        g = one_hot - probs
        sqrt_probs = torch.sqrt(probs + self.eps)
        t = 0.5 * g / sqrt_probs

        probs_clamped = torch.clamp(probs, min=self.eps)
        u = torch.sqrt(probs_clamped)
        u = u / (torch.norm(u, dim=-1, keepdim=True) + self.eps)

        v_parallel = torch.sum(t * u, dim=-1, keepdim=True) * u
        t_tangent = t - v_parallel

        belief_norms = torch.norm(t_tangent, dim=-1)
        return float(belief_norms.mean().cpu())

    def analyze_prompt_at_layer(
        self,
        model,
        tokenizer,
        prompt: str,
        layer_idx: int
    ) -> Dict[str, float]:
        """Analyze nDNA metrics for a prompt at a specific layer."""
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=128,
            padding=False
        ).to(self.device)

        with torch.no_grad():
            outputs = model(
                **inputs,
                output_hidden_states=True,
                return_dict=True
            )

        if layer_idx >= len(outputs.hidden_states):
            layer_idx = len(outputs.hidden_states) - 1

        hidden = outputs.hidden_states[layer_idx].squeeze(0)

        # Get lm_head
        if hasattr(model, 'lm_head'):
            lm_head = model.lm_head
        elif hasattr(model, 'base_model'):
            lm_head = model.base_model.lm_head
        else:
            lm_head = model.model.lm_head

        return {
            'spectral': self.compute_spectral_curvature(hidden),
            'thermo': self.compute_thermodynamic_length(hidden, lm_head),
            'belief': self.compute_belief_vector(hidden, lm_head),
        }

    def analyze_model(
        self,
        model,
        tokenizer,
        prompts: List[str],
        layer_indices: List[int],
        desc: str = "Analyzing"
    ) -> Dict[str, np.ndarray]:
        """Analyze model across all prompts and layers."""
        results = {
            'spectral': {l: [] for l in layer_indices},
            'thermo': {l: [] for l in layer_indices},
            'belief': {l: [] for l in layer_indices},
        }

        for prompt in tqdm(prompts, desc=desc):
            for layer_idx in layer_indices:
                try:
                    metrics = self.analyze_prompt_at_layer(
                        model, tokenizer, prompt, layer_idx
                    )
                    results['spectral'][layer_idx].append(metrics['spectral'])
                    results['thermo'][layer_idx].append(metrics['thermo'])
                    results['belief'][layer_idx].append(metrics['belief'])
                except Exception as e:
                    continue

        # Convert to arrays
        return {
            'layers': np.array(layer_indices),
            'spectral': np.array([np.mean(results['spectral'][l]) if results['spectral'][l] else 0
                                  for l in layer_indices]),
            'thermo': np.array([np.mean(results['thermo'][l]) if results['thermo'][l] else 0
                               for l in layer_indices]),
            'belief': np.array([np.mean(results['belief'][l]) if results['belief'][l] else 0
                               for l in layer_indices]),
        }


model_ndna = ModelNDNA(device=DEVICE)
print("✅ Model nDNA Calculator ready")

✅ Model nDNA Calculator ready


In [10]:
# ============================================================================
# CELL 8: MODEL LOADING FUNCTIONS
# ============================================================================

def load_model(
    model_id: str,
    adapter_path: Optional[str] = None,
    name: str = "Model"
) -> Tuple[Any, Any]:
    """Load model with optional adapter."""
    print(f"\n{'='*60}")
    print(f"📥 Loading {name}...")
    print(f"{'='*60}")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=COMPUTE_DTYPE,
    )

    if adapter_path and os.path.exists(adapter_path):
        adapter_config = os.path.join(adapter_path, "adapter_config.json")
        if os.path.exists(adapter_config):
            print(f"   Loading adapter: {adapter_path}")
            model = PeftModel.from_pretrained(model, adapter_path)
            model = model.merge_and_unload()
            print(f"   ✅ Adapter merged successfully")
        else:
            print(f"   ⚠️ No adapter_config.json found")

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model.eval()

    num_layers = model.config.num_hidden_layers
    print(f"   ✅ {name}: {num_layers} layers, {model.config.hidden_size}d embeddings")

    return model, tokenizer


def load_model_full_precision(
    model_id: str,
    adapter_path: Optional[str] = None,
    name: str = "Model"
) -> Any:
    """Load model in full precision for merging."""
    print(f"\n📥 Loading {name} (full precision)...")

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

    if adapter_path and os.path.exists(adapter_path):
        adapter_config = os.path.join(adapter_path, "adapter_config.json")
        if os.path.exists(adapter_config):
            model = PeftModel.from_pretrained(model, adapter_path)
            model = model.merge_and_unload()

    return model


print("✅ Model loading functions ready")

✅ Model loading functions ready


In [11]:
# ============================================================================
# CELL 9: FISHER MERGING WITH VALIDATION
# ============================================================================

def fisher_merge_models(
    base_model_id: str,
    adapter1_path: str,
    adapter2_path: str,
    output_path: str,
    alpha: float = 0.5,
    validate: bool = True
) -> Tuple[Any, Any, Dict]:
    """
    Merge two fine-tuned models using Fisher-weighted averaging.

    Returns: (merged_model, tokenizer, validation_metrics)
    """
    print("\n" + "=" * 70)
    print("🧬 FISHER MERGING: Creating Offspring Model")
    print("=" * 70)
    print(f"   Alpha (African weight): {alpha}")
    print(f"   Beta (Latin weight): {1 - alpha}")

    validation_metrics = {}

    # Load base model state
    print("\n📥 Loading base model state...")
    base_model = load_model_full_precision(base_model_id, None, "Base")
    base_state = {k: v.clone().cpu() for k, v in base_model.state_dict().items()}
    del base_model
    clear_memory()

    # Load model 1 (African) state
    print("📥 Loading African model state...")
    model1 = load_model_full_precision(base_model_id, adapter1_path, "African")
    state1 = {k: v.clone().cpu() for k, v in model1.state_dict().items()}
    del model1
    clear_memory()

    # Load model 2 (Latin) state
    print("📥 Loading Latin model state...")
    model2 = load_model_full_precision(base_model_id, adapter2_path, "Latin")
    state2 = {k: v.clone().cpu() for k, v in model2.state_dict().items()}
    del model2
    clear_memory()

    # Compute deltas and merge
    print("\n🔀 Computing Fisher merge...")
    merged_state = {}
    delta_norms = {'african': [], 'latin': [], 'merged': []}

    for key in tqdm(base_state.keys(), desc="Merging"):
        if key in state1 and key in state2:
            # Compute deltas from base
            delta1 = state1[key].float() - base_state[key].float()
            delta2 = state2[key].float() - base_state[key].float()

            # Fisher-weighted merge: base + α*Δ1 + (1-α)*Δ2
            merged_delta = alpha * delta1 + (1 - alpha) * delta2
            merged_state[key] = base_state[key].float() + merged_delta

            # Track delta norms for validation
            if 'weight' in key and delta1.numel() > 1000:
                delta_norms['african'].append(float(torch.norm(delta1)))
                delta_norms['latin'].append(float(torch.norm(delta2)))
                delta_norms['merged'].append(float(torch.norm(merged_delta)))
        else:
            merged_state[key] = base_state[key]

    # Validation metrics
    validation_metrics['mean_delta_norm'] = {
        'african': np.mean(delta_norms['african']),
        'latin': np.mean(delta_norms['latin']),
        'merged': np.mean(delta_norms['merged']),
    }

    print(f"\n📊 Merge Validation:")
    print(f"   African mean delta norm: {validation_metrics['mean_delta_norm']['african']:.4f}")
    print(f"   Latin mean delta norm: {validation_metrics['mean_delta_norm']['latin']:.4f}")
    print(f"   Merged mean delta norm: {validation_metrics['mean_delta_norm']['merged']:.4f}")

    # Expected: merged ≈ α * african + (1-α) * latin
    expected_merged = alpha * validation_metrics['mean_delta_norm']['african'] + \
                      (1-alpha) * validation_metrics['mean_delta_norm']['latin']
    actual_merged = validation_metrics['mean_delta_norm']['merged']
    validation_metrics['merge_accuracy'] = 1.0 - abs(expected_merged - actual_merged) / (expected_merged + 1e-10)

    print(f"   Expected merged norm: {expected_merged:.4f}")
    print(f"   Actual merged norm: {actual_merged:.4f}")
    print(f"   Merge accuracy: {validation_metrics['merge_accuracy']*100:.2f}%")

    if validation_metrics['merge_accuracy'] < 0.8:
        print("   ⚠️ WARNING: Merge accuracy below 80%!")
    else:
        print("   ✅ Merge validated successfully!")

    # Load merged state into model
    print("\n💾 Saving merged model...")
    model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        device_map="cpu",
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

    # Convert merged state to correct dtype
    for key in merged_state:
        merged_state[key] = merged_state[key].to(model.state_dict()[key].dtype)

    model.load_state_dict(merged_state)

    # Save
    os.makedirs(output_path, exist_ok=True)
    model.save_pretrained(output_path)

    tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.save_pretrained(output_path)

    print(f"   ✅ Saved to: {output_path}")

    # Clean up
    del model, base_state, state1, state2, merged_state
    clear_memory()

    # Reload with quantization
    merged_model, tokenizer = load_model(output_path, None, "Offspring Model")

    return merged_model, tokenizer, validation_metrics


print("✅ Fisher merge function ready")

✅ Fisher merge function ready


In [12]:
# ============================================================================
# CELL 10: LOAD BASE MODEL
# ============================================================================

base_model, tokenizer = load_model(
    config.base_model_id,
    adapter_path=None,
    name="allenai/Llama-3.1-Tulu-3.1-8B" #"meta-llama/Llama-3.2-3B-Instruct"
)

NUM_LAYERS = base_model.config.num_hidden_layers
HIDDEN_DIM = base_model.config.hidden_size
ANALYSIS_LAYERS = list(range(1, NUM_LAYERS + 1))  # All layers including embeddings

print(f"\n📊 Model Configuration:")
print(f"   Layers: {NUM_LAYERS}")
print(f"   Hidden dim: {HIDDEN_DIM}")
print(f"   Analyzing layers: 1 to {NUM_LAYERS}")


📥 Loading allenai/Llama-3.1-Tulu-3.1-8B...


config.json:   0%|          | 0.00/895 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/180 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

   ✅ allenai/Llama-3.1-Tulu-3.1-8B: 32 layers, 4096d embeddings

📊 Model Configuration:
   Layers: 32
   Hidden dim: 4096
   Analyzing layers: 1 to 32


In [17]:
# ============================================================================
# CELL 11: BASE MODEL - WORD EMBEDDING EXTRACTION
# ============================================================================

print("\n" + "=" * 70)
print("📝 BASE MODEL: PURE WORD EMBEDDING EXTRACTION")
print("=" * 70)

base_word_embeddings = word_extractor.extract_all_words_all_layers(
    base_model,
    tokenizer,
    ALL_WORDS,
    desc="Base Model Words"
)

# Sanity check: verify words have DIFFERENT embeddings
print("\n🔍 SANITY CHECK: Word Embedding Differences")
print("-" * 50)

test_pairs = [
    ("war", "peace"),      # Should be DIFFERENT
    ("war", "destroy"),    # Should be SIMILAR
    ("peace", "love"),     # Should be SIMILAR
    ("justice", "freedom"), # Should be SIMILAR
]

last_layer = NUM_LAYERS

for w1, w2 in test_pairs:
    emb1 = base_word_embeddings[w1][last_layer]
    emb2 = base_word_embeddings[w2][last_layer]
    sim = cosine_similarity(emb1, emb2)

    expected = "DIFFERENT" if w1 in ["war", "destroy"] and w2 in ["peace", "love"] else "SIMILAR"
    actual = "SIMILAR" if sim > 0.5 else "DIFFERENT"
    status = ">>" if expected == actual else ">>"

    print(f"   {status} {w1:10s} vs {w2:10s}: sim={sim:.4f} ({expected} category)")


📝 BASE MODEL: PURE WORD EMBEDDING EXTRACTION


Base Model Words:   0%|          | 0/29 [00:00<?, ?it/s]


🔍 SANITY CHECK: Word Embedding Differences
--------------------------------------------------
   >> war        vs peace     : sim=0.4662 (DIFFERENT category)
   >> war        vs destroy   : sim=0.4449 (SIMILAR category)
   >> peace      vs love      : sim=0.5552 (SIMILAR category)
   >> justice    vs freedom   : sim=0.5549 (SIMILAR category)


In [18]:
# ============================================================================
# CELL 12: BASE MODEL - SIMILARITY MATRIX
# ============================================================================

print("\n📊 Computing similarity matrices...")

# Compute at multiple layers
layer_samples = [0, NUM_LAYERS // 4, NUM_LAYERS // 2, 3 * NUM_LAYERS // 4, NUM_LAYERS]

base_sim_matrices = {}
for layer_idx in layer_samples:
    if layer_idx <= NUM_LAYERS:
        base_sim_matrices[layer_idx] = word_extractor.compute_similarity_matrix(
            base_word_embeddings, ALL_WORDS, layer_idx
        )

# Show last layer matrix
print(f"\n📊 Similarity Matrix at Layer {NUM_LAYERS} (Last Layer):")
print("-" * 50)

# Create nice display
sim_df = pd.DataFrame(
    base_sim_matrices[NUM_LAYERS],
    index=ALL_WORDS,
    columns=ALL_WORDS
)
print(sim_df.round(2).to_string())

# Save to CSV
save_csv(sim_df, "base_similarity_matrix_last_layer.csv")


📊 Computing similarity matrices...

📊 Similarity Matrix at Layer 32 (Last Layer):
--------------------------------------------------
             destroy   war  protest  violence  attack  peace  love  harmony  unity  cooperation  justice  freedom  wisdom  truth  honor  tradition  belief  culture  ritual  custom  concept  idea  thought  reason  logic  skill  hardwork  motivation  understand
destroy         1.00  0.44     0.56      0.52    0.74   0.46  0.52     0.56   0.56         0.55     0.54     0.49    0.54   0.57   0.50       0.56    0.62     0.61    0.54    0.53     0.61  0.46     0.46    0.64   0.50   0.53      0.60        0.56        0.70
war             0.44  1.00     0.49      0.47    0.51   0.47  0.40     0.56   0.38         0.43     0.49     0.47    0.47   0.47   0.44       0.48    0.46     0.50    0.46    0.33     0.45  0.34     0.37    0.42   0.40   0.41      0.47        0.47        0.40
protest         0.56  0.49     1.00      0.56    0.60   0.50  0.48     0.59   0.50    

'/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/base_similarity_matrix_last_layer.csv'

In [19]:
# ============================================================================
# CELL 13: BASE MODEL - nDNA ANALYSIS (PROMPTS)
# ============================================================================

print("\n" + "=" * 70)
print("🧬 BASE MODEL: nDNA ANALYSIS (Socio-Cultural Prompts)")
print("=" * 70)

base_ndna = model_ndna.analyze_model(
    base_model,
    tokenizer,
    SOCIO_PROBES,
    ANALYSIS_LAYERS,
    desc="Base Model nDNA"
)

print(f"\n📊 BASE MODEL nDNA SUMMARY:")
print(f"   Spectral κ: {base_ndna['spectral'].mean():.4f} ± {base_ndna['spectral'].std():.4f}")
print(f"   Thermo Δ:   {base_ndna['thermo'].mean():.4f} ± {base_ndna['thermo'].std():.4f}")
print(f"   Belief β:   {base_ndna['belief'].mean():.4f} ± {base_ndna['belief'].std():.4f}")

# Save to CSV
base_ndna_df = pd.DataFrame({
    'layer': base_ndna['layers'],
    'spectral': base_ndna['spectral'],
    'thermo': base_ndna['thermo'],
    'belief': base_ndna['belief'],
})
save_csv(base_ndna_df, "base_ndna_by_layer.csv")


🧬 BASE MODEL: nDNA ANALYSIS (Socio-Cultural Prompts)


Base Model nDNA:   0%|          | 0/54 [00:00<?, ?it/s]


📊 BASE MODEL nDNA SUMMARY:
   Spectral κ: 0.7472 ± 0.4225
   Thermo Δ:   5.4223 ± 3.8000
   Belief β:   84.1026 ± 51.5405
💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/base_ndna_by_layer.csv


'/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/base_ndna_by_layer.csv'

In [20]:
# ============================================================================
# CELL 14: CLEAR BASE MODEL, LOAD AFRICAN MODEL
# ============================================================================

del base_model
clear_memory()

african_model = None
african_word_embeddings = None
african_ndna = None
african_sim_matrices = None

if os.path.exists(config.african_adapter):
    african_model, _ = load_model(
        config.base_model_id,
        adapter_path=config.african_adapter,
        name="African Cultural Model"
    )

    # Word embeddings
    print("\n📝 AFRICAN MODEL: Word Embedding Extraction")
    african_word_embeddings = word_extractor.extract_all_words_all_layers(
        african_model,
        tokenizer,
        ALL_WORDS,
        desc="African Model Words"
    )

    # Similarity matrices
    african_sim_matrices = {}
    for layer_idx in layer_samples:
        if layer_idx <= NUM_LAYERS:
            african_sim_matrices[layer_idx] = word_extractor.compute_similarity_matrix(
                african_word_embeddings, ALL_WORDS, layer_idx
            )

    # nDNA
    print("\n🧬 AFRICAN MODEL: nDNA Analysis")
    african_ndna = model_ndna.analyze_model(
        african_model,
        tokenizer,
        SOCIO_PROBES,
        ANALYSIS_LAYERS,
        desc="African Model nDNA"
    )

    print(f"\n📊 AFRICAN MODEL SUMMARY:")
    print(f"   Spectral κ: {african_ndna['spectral'].mean():.4f}")
    print(f"   Thermo Δ:   {african_ndna['thermo'].mean():.4f}")
    print(f"   Belief β:   {african_ndna['belief'].mean():.4f}")

    african_ndna_df = pd.DataFrame({
        'layer': african_ndna['layers'],
        'spectral': african_ndna['spectral'],
        'thermo': african_ndna['thermo'],
        'belief': african_ndna['belief'],
    })
    save_csv(african_ndna_df, "african_ndna_by_layer.csv")
else:
    print(f"⚠️ African adapter not found at: {config.african_adapter}")


📥 Loading African Cultural Model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

   Loading adapter: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/africa_adapter
   ✅ Adapter merged successfully
   ✅ African Cultural Model: 32 layers, 4096d embeddings

📝 AFRICAN MODEL: Word Embedding Extraction


African Model Words:   0%|          | 0/29 [00:00<?, ?it/s]


🧬 AFRICAN MODEL: nDNA Analysis


African Model nDNA:   0%|          | 0/54 [00:00<?, ?it/s]


📊 AFRICAN MODEL SUMMARY:
   Spectral κ: 0.7233
   Thermo Δ:   5.3362
   Belief β:   87.4961
💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/african_ndna_by_layer.csv


In [36]:
ls

checkpoint-3750/  merged_offspring_model/
latin_adapter/    ndna_validated_results/


In [21]:
# ============================================================================
# CELL 15: CLEAR AFRICAN MODEL, LOAD LATIN MODEL
# ============================================================================

if african_model is not None:
    del african_model
    clear_memory()

latin_model = None
latin_word_embeddings = None
latin_ndna = None
latin_sim_matrices = None

if os.path.exists(config.latin_adapter):
    latin_model, _ = load_model(
        config.base_model_id,
        adapter_path=config.latin_adapter,
        name="Latin American Cultural Model"
    )

    # Word embeddings
    print("\n📝 LATIN MODEL: Word Embedding Extraction")
    latin_word_embeddings = word_extractor.extract_all_words_all_layers(
        latin_model,
        tokenizer,
        ALL_WORDS,
        desc="Latin Model Words"
    )

    # Similarity matrices
    latin_sim_matrices = {}
    for layer_idx in layer_samples:
        if layer_idx <= NUM_LAYERS:
            latin_sim_matrices[layer_idx] = word_extractor.compute_similarity_matrix(
                latin_word_embeddings, ALL_WORDS, layer_idx
            )

    # nDNA
    print("\n🧬 LATIN MODEL: nDNA Analysis")
    latin_ndna = model_ndna.analyze_model(
        latin_model,
        tokenizer,
        SOCIO_PROBES,
        ANALYSIS_LAYERS,
        desc="Latin Model nDNA"
    )

    print(f"\n📊 LATIN MODEL SUMMARY:")
    print(f"   Spectral κ: {latin_ndna['spectral'].mean():.4f}")
    print(f"   Thermo Δ:   {latin_ndna['thermo'].mean():.4f}")
    print(f"   Belief β:   {latin_ndna['belief'].mean():.4f}")

    latin_ndna_df = pd.DataFrame({
        'layer': latin_ndna['layers'],
        'spectral': latin_ndna['spectral'],
        'thermo': latin_ndna['thermo'],
        'belief': latin_ndna['belief'],
    })
    save_csv(latin_ndna_df, "latin_ndna_by_layer.csv")
else:
    print(f"⚠️ Latin adapter not found at: {config.latin_adapter}")


📥 Loading Latin American Cultural Model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

   Loading adapter: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/latin_adapter
   ✅ Adapter merged successfully
   ✅ Latin American Cultural Model: 32 layers, 4096d embeddings

📝 LATIN MODEL: Word Embedding Extraction


Latin Model Words:   0%|          | 0/29 [00:00<?, ?it/s]


🧬 LATIN MODEL: nDNA Analysis


Latin Model nDNA:   0%|          | 0/54 [00:00<?, ?it/s]


📊 LATIN MODEL SUMMARY:
   Spectral κ: 0.7485
   Thermo Δ:   5.3568
   Belief β:   88.0856
💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/latin_ndna_by_layer.csv


In [22]:
# ============================================================================
# CELL 16: CREATE OFFSPRING MODEL (FISHER MERGE)
# ============================================================================

if latin_model is not None:
    del latin_model
    clear_memory()

offspring_model = None
offspring_word_embeddings = None
offspring_ndna = None
offspring_sim_matrices = None
merge_validation = None

if os.path.exists(config.african_adapter) and os.path.exists(config.latin_adapter):
    offspring_model, _, merge_validation = fisher_merge_models(
        config.base_model_id,
        config.african_adapter,
        config.latin_adapter,
        config.merged_output,
        alpha=0.5,
        validate=True
    )

    # Word embeddings
    print("\n📝 OFFSPRING MODEL: Word Embedding Extraction")
    offspring_word_embeddings = word_extractor.extract_all_words_all_layers(
        offspring_model,
        tokenizer,
        ALL_WORDS,
        desc="Offspring Model Words"
    )

    # Similarity matrices
    offspring_sim_matrices = {}
    for layer_idx in layer_samples:
        if layer_idx <= NUM_LAYERS:
            offspring_sim_matrices[layer_idx] = word_extractor.compute_similarity_matrix(
                offspring_word_embeddings, ALL_WORDS, layer_idx
            )

    # nDNA
    print("\n🧬 OFFSPRING MODEL: nDNA Analysis")
    offspring_ndna = model_ndna.analyze_model(
        offspring_model,
        tokenizer,
        SOCIO_PROBES,
        ANALYSIS_LAYERS,
        desc="Offspring Model nDNA"
    )

    print(f"\n📊 OFFSPRING MODEL SUMMARY:")
    print(f"   Spectral κ: {offspring_ndna['spectral'].mean():.4f}")
    print(f"   Thermo Δ:   {offspring_ndna['thermo'].mean():.4f}")
    print(f"   Belief β:   {offspring_ndna['belief'].mean():.4f}")

    offspring_ndna_df = pd.DataFrame({
        'layer': offspring_ndna['layers'],
        'spectral': offspring_ndna['spectral'],
        'thermo': offspring_ndna['thermo'],
        'belief': offspring_ndna['belief'],
    })
    save_csv(offspring_ndna_df, "offspring_ndna_by_layer.csv")
else:
    print("⚠️ Cannot create offspring - missing one or both adapters")


🧬 FISHER MERGING: Creating Offspring Model
   Alpha (African weight): 0.5
   Beta (Latin weight): 0.5

📥 Loading base model state...

📥 Loading Base (full precision)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

📥 Loading African model state...

📥 Loading African (full precision)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

📥 Loading Latin model state...

📥 Loading Latin (full precision)...


...

   ✅ Offspring Model: 32 layers, 4096d embeddings

📝 OFFSPRING MODEL: Word Embedding Extraction


Offspring Model Words:   0%|          | 0/29 [00:00<?, ?it/s]


🧬 OFFSPRING MODEL: nDNA Analysis


Offspring Model nDNA:   0%|          | 0/54 [00:00<?, ?it/s]


📊 OFFSPRING MODEL SUMMARY:
   Spectral κ: 0.7233
   Thermo Δ:   5.3111
   Belief β:   88.9704
💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/offspring_ndna_by_layer.csv


In [23]:
# ============================================================================
# CELL 17: VISUALIZATION SETUP
# ============================================================================

pio.renderers.default = "notebook"

MODEL_COLORS = {
    'Base': '#2E86AB',           # Blue
    'African': '#F18F01',        # Orange
    'Latin': '#7B2D8E',          # Purple
    'Offspring': '#2D8E4F',      # Green
}

CATEGORY_COLORS = {
    'conflict': '#E63946',       # Red
    'harmony': '#2A9D8F',        # Teal
    'virtue': '#7209B7',         # Purple
    'culture': '#E9C46A',        # Yellow
    'abstract': '#457B9D',       # Blue
}

print("✅ Visualization ready")
print(f"   Output directory: {config.output_dir}")

✅ Visualization ready
   Output directory: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/


In [38]:
# ============================================================================
# CELL 46: SETUP PLOTLY FOR COLAB DISPLAY
# ============================================================================

print("\n" + "=" * 70)
print("📊 SETTING UP PLOTLY FOR GOOGLE COLAB")
print("=" * 70)

import plotly.io as pio
from IPython.display import display, HTML

# Force Colab renderer
pio.renderers.default = 'colab'

# Alternative: Use notebook renderer
# pio.renderers.default = 'notebook'

# Function to forcefully display plots in Colab
def show_plot(fig, filename=None):
    """
    Forcefully display plotly figure in Google Colab.
    Also saves to HTML file if filename provided.
    """
    # Save if filename provided
    if filename:
        filepath = os.path.join(config.output_dir, filename)
        fig.write_html(filepath, include_plotlyjs='cdn')
        print(f"💾 Saved: {filename}")

    # Method 1: Direct show with colab renderer
    try:
        fig.show(renderer='colab')
    except:
        pass

    # Method 2: Display as HTML (backup)
    try:
        display(HTML(fig.to_html(include_plotlyjs='cdn')))
    except:
        pass

    # Method 3: Use iframe display
    try:
        from IPython.display import IFrame
        import tempfile
        with tempfile.NamedTemporaryFile(suffix='.html', delete=False) as f:
            fig.write_html(f.name, include_plotlyjs='cdn')
            display(IFrame(f.name, width=1000, height=600))
    except:
        pass

# Test display
test_fig = go.Figure()
test_fig.add_trace(go.Scatter(x=[1,2,3], y=[1,2,3], mode='markers+lines', name='Test'))
test_fig.update_layout(title="Test Plot - If you see this, Plotly is working!")

print("\n🧪 Testing Plotly display...")
show_plot(test_fig)
print("✅ If you see the test plot above, display is working!")


📊 SETTING UP PLOTLY FOR GOOGLE COLAB

🧪 Testing Plotly display...


✅ If you see the test plot above, display is working!


In [39]:
# ============================================================================
# CELL 18: PLOT 1 - ALL MODELS nDNA COMPARISON (3 metrics)
# ============================================================================

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=(
        '📊 Spectral Curvature (κ) - Information Distribution',
        '📊 Thermodynamic Length (Δ) - Probability Distance',
        '📊 Belief Vector (β) - Gradient Magnitude'
    ),
    vertical_spacing=0.08
)

# Data to plot
plot_data = [
    ('Base', base_ndna, '-'),
]
if african_ndna is not None:
    plot_data.append(('African', african_ndna, '-'))
if latin_ndna is not None:
    plot_data.append(('Latin', latin_ndna, '-'))
if offspring_ndna is not None:
    plot_data.append(('Offspring', offspring_ndna, 'dash'))

for name, data, dash in plot_data:
    layers = data['layers']

    # Spectral
    fig.add_trace(go.Scatter(
        x=layers, y=data['spectral'],
        mode='lines+markers', name=name,
        line=dict(color=MODEL_COLORS[name], width=2, dash=dash if dash != '-' else None),
        marker=dict(size=4),
        legendgroup=name,
        showlegend=True
    ), row=1, col=1)

    # Thermo
    fig.add_trace(go.Scatter(
        x=layers, y=data['thermo'],
        mode='lines+markers', name=name,
        line=dict(color=MODEL_COLORS[name], width=2, dash=dash if dash != '-' else None),
        marker=dict(size=4),
        legendgroup=name,
        showlegend=False
    ), row=2, col=1)

    # Belief
    fig.add_trace(go.Scatter(
        x=layers, y=data['belief'],
        mode='lines+markers', name=name,
        line=dict(color=MODEL_COLORS[name], width=2, dash=dash if dash != '-' else None),
        marker=dict(size=4),
        legendgroup=name,
        showlegend=False
    ), row=3, col=1)

fig.update_xaxes(title_text="Layer", row=3, col=1)
fig.update_yaxes(title_text="κ", row=1, col=1)
fig.update_yaxes(title_text="Δ", row=2, col=1)
fig.update_yaxes(title_text="β", row=3, col=1)

fig.update_layout(
    title=dict(
        text="🧬 nDNA Metrics: All Models Comparison",
        font=dict(size=20)
    ),
    height=900,
    width=1000,
    template='plotly_white',
    legend=dict(x=0.85, y=0.98, font=dict(size=11)),
)
fig.show()
save_figure(fig, "01_all_models_ndna_comparison.html")

💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/01_all_models_ndna_comparison.html


'/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/01_all_models_ndna_comparison.html'

In [40]:
# ============================================================================
# CELL 19: PLOT 2 - 3D TRAJECTORY: All Models
# ============================================================================

fig = go.Figure()

for name, data, _ in plot_data:
    fig.add_trace(go.Scatter3d(
        x=data['layers'],
        y=data['spectral'],
        z=data['belief'],
        mode='lines+markers',
        name=name,
        line=dict(color=MODEL_COLORS[name], width=5),
        marker=dict(size=4),
    ))

fig.update_layout(
    title=dict(
        text="🧬 3D nDNA Trajectory: Layer × Spectral × Belief",
        font=dict(size=18)
    ),
    scene=dict(
        xaxis_title="Layer",
        yaxis_title="Spectral κ",
        zaxis_title="Belief β",
    ),
    height=700,
    width=900,
    template='plotly_white',
)
fig.show()
save_figure(fig, "02_3d_ndna_trajectory.html")

💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/02_3d_ndna_trajectory.html


'/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/02_3d_ndna_trajectory.html'

In [41]:
# ============================================================================
# CELL 20: PLOT 3 - WORD SIMILARITY HEATMAPS (Base Model)
# ============================================================================

# Create subplot with heatmaps at different layers
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f'Layer {l}' for l in [0, NUM_LAYERS//2, NUM_LAYERS]],
    horizontal_spacing=0.08
)

for col, layer_idx in enumerate([0, NUM_LAYERS//2, NUM_LAYERS], 1):
    if layer_idx in base_sim_matrices:
        fig.add_trace(go.Heatmap(
            z=base_sim_matrices[layer_idx],
            x=ALL_WORDS,
            y=ALL_WORDS,
            colorscale='RdBu',
            zmid=0,
            zmin=-1, zmax=1,
            showscale=(col == 3),
            colorbar=dict(title="Similarity", x=1.02) if col == 3 else None,
        ), row=1, col=col)

fig.update_layout(
    title=dict(
        text="🔍 Word Similarity Evolution Across Layers (Base Model)",
        font=dict(size=18)
    ),
    height=500,
    width=1400,
    template='plotly_white',
)

# Rotate x-axis labels
fig.update_xaxes(tickangle=45)

save_figure(fig, "03_word_similarity_heatmaps_base.html")

💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/03_word_similarity_heatmaps_base.html


'/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/03_word_similarity_heatmaps_base.html'

In [42]:
# ============================================================================
# CELL 21: PLOT 4 - DETAILED SIMILARITY HEATMAP (Last Layer, All Models)
# ============================================================================

# Determine how many models we have
available_models = [('Base', base_sim_matrices)]
if african_sim_matrices:
    available_models.append(('African', african_sim_matrices))
if latin_sim_matrices:
    available_models.append(('Latin', latin_sim_matrices))
if offspring_sim_matrices:
    available_models.append(('Offspring', offspring_sim_matrices))

fig = make_subplots(
    rows=1, cols=len(available_models),
    subplot_titles=[name for name, _ in available_models],
    horizontal_spacing=0.05
)

for col, (name, sim_dict) in enumerate(available_models, 1):
    if NUM_LAYERS in sim_dict:
        # Add annotations for values
        text_matrix = np.round(sim_dict[NUM_LAYERS], 2).astype(str)

        fig.add_trace(go.Heatmap(
            z=sim_dict[NUM_LAYERS],
            x=ALL_WORDS,
            y=ALL_WORDS,
            colorscale='RdBu',
            zmid=0,
            zmin=-0.5, zmax=1,
            showscale=(col == len(available_models)),
            text=text_matrix,
            texttemplate="%{text}",
            textfont={"size": 7},
            colorbar=dict(title="Cosine Sim", x=1.02) if col == len(available_models) else None,
        ), row=1, col=col)

fig.update_layout(
    title=dict(
        text=f"🔍 Word Similarity Comparison at Layer {NUM_LAYERS} (All Models)",
        font=dict(size=18)
    ),
    height=600,
    width=400 * len(available_models),
    template='plotly_white',
)

fig.update_xaxes(tickangle=45, tickfont=dict(size=8))
fig.update_yaxes(tickfont=dict(size=8))
fig.show()
save_figure(fig, "04_word_similarity_all_models_last_layer.html")

💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/04_word_similarity_all_models_last_layer.html


'/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/04_word_similarity_all_models_last_layer.html'

In [49]:
ALL_WORDS

['destroy',
 'war',
 'protest',
 'violence',
 'attack',
 'peace',
 'love',
 'harmony',
 'unity',
 'cooperation',
 'justice',
 'freedom',
 'wisdom',
 'truth',
 'honor',
 'tradition',
 'belief',
 'culture',
 'ritual',
 'custom',
 'concept',
 'idea',
 'thought',
 'reason',
 'logic',
 'skill',
 'hardwork',
 'motivation',
 'understand']

In [43]:
# ============================================================================
# CELL 22: PLOT 5 - WORD EMBEDDING NORM TRAJECTORIES
# ============================================================================

fig = go.Figure()

# Calculate embedding norms per layer for each word
for word in ALL_WORDS:
    layers = sorted(base_word_embeddings[word].keys())
    norms = [np.linalg.norm(base_word_embeddings[word][l]) for l in layers]

    fig.add_trace(go.Scatter(
        x=layers,
        y=norms,
        mode='lines',
        name=word.capitalize(),
        line=dict(color=WORD_TO_COLOR[word], width=2),
    ))

fig.update_layout(
    title=dict(
        text="📐 Word Embedding Norm by Layer (Base Model)",
        font=dict(size=18)
    ),
    xaxis_title="Layer",
    yaxis_title="Embedding Norm ||e||",
    height=600,
    width=1000,
    template='plotly_white',
    legend=dict(x=1.02, y=0.98, font=dict(size=9)),
)
fig.show()
save_figure(fig, "05_word_embedding_norm_base.html")

💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/05_word_embedding_norm_base.html


'/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/05_word_embedding_norm_base.html'

In [44]:
# ============================================================================
# CELL 23: PLOT 6 - SEMANTIC CATEGORY ANALYSIS (Avg similarity within/across)
# ============================================================================

def compute_category_similarities(sim_matrix, words, categories):
    """Compute within-category and across-category similarities."""
    results = {
        'within': {},
        'across': {}
    }

    cat_indices = {}
    for cat, info in categories.items():
        indices = [i for i, w in enumerate(words) if w in info['words']]
        cat_indices[cat] = indices

    # Within-category similarity
    for cat, indices in cat_indices.items():
        if len(indices) > 1:
            sims = []
            for i in range(len(indices)):
                for j in range(i+1, len(indices)):
                    sims.append(sim_matrix[indices[i], indices[j]])
            results['within'][cat] = np.mean(sims) if sims else 0

    # Across-category similarity (conflict vs harmony)
    if 'conflict' in cat_indices and 'harmony' in cat_indices:
        sims = []
        for i in cat_indices['conflict']:
            for j in cat_indices['harmony']:
                sims.append(sim_matrix[i, j])
        results['across']['conflict_vs_harmony'] = np.mean(sims) if sims else 0

    return results

# Compute for base model
base_cat_sims = compute_category_similarities(
    base_sim_matrices[NUM_LAYERS], ALL_WORDS, WORD_CATEGORIES
)

print("📊 Category Similarity Analysis (Base Model):")
print("-" * 50)
print("\n Within-Category (should be HIGH):")
for cat, sim in base_cat_sims['within'].items():
    print(f"   {cat}: {sim:.4f}")

print("\n Across-Category (conflict vs harmony - should be LOW/NEGATIVE):")
for pair, sim in base_cat_sims['across'].items():
    print(f"   {pair}: {sim:.4f}")

# Create bar chart
fig = go.Figure()

# Within-category
cats = list(base_cat_sims['within'].keys())
vals = list(base_cat_sims['within'].values())
colors = [CATEGORY_COLORS[c] for c in cats]

fig.add_trace(go.Bar(
    x=cats,
    y=vals,
    name='Within-Category',
    marker_color=colors,
    text=[f'{v:.3f}' for v in vals],
    textposition='outside',
))

# Add across-category as separate bar
if 'conflict_vs_harmony' in base_cat_sims['across']:
    fig.add_trace(go.Bar(
        x=['Conflict vs Harmony'],
        y=[base_cat_sims['across']['conflict_vs_harmony']],
        name='Across-Category',
        marker_color='#666666',
        text=[f"{base_cat_sims['across']['conflict_vs_harmony']:.3f}"],
        textposition='outside',
    ))

fig.update_layout(
    title=dict(
        text="📊 Semantic Category Similarity Analysis (Base Model, Last Layer)",
        font=dict(size=18)
    ),
    xaxis_title="Category",
    yaxis_title="Average Cosine Similarity",
    height=500,
    width=800,
    template='plotly_white',
    showlegend=False,
)
fig.show()
save_figure(fig, "06_category_similarity_analysis.html")

📊 Category Similarity Analysis (Base Model):
--------------------------------------------------

 Within-Category (should be HIGH):
   conflict: 0.5478
   harmony: 0.5191
   virtue: 0.5794
   culture: 0.5637
   abstract: 0.5376

 Across-Category (conflict vs harmony - should be LOW/NEGATIVE):
   conflict_vs_harmony: 0.5094


💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/06_category_similarity_analysis.html


'/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/06_category_similarity_analysis.html'

In [45]:
# ============================================================================
# CELL 24: PLOT 7 - KEY WORD PAIR COMPARISON ACROSS MODELS
# ============================================================================

# Track specific word pairs across models
key_pairs = [
    ("war", "peace"),
    ("war", "destroy"),
    ("peace", "love"),
    ("justice", "freedom"),
    ("culture", "tradition"),
]

# Compute similarities for each model at last layer
pair_sims = {pair: {} for pair in key_pairs}

for w1, w2 in key_pairs:
    i1 = ALL_WORDS.index(w1)
    i2 = ALL_WORDS.index(w2)

    pair_sims[(w1, w2)]['Base'] = base_sim_matrices[NUM_LAYERS][i1, i2]

    if african_sim_matrices:
        pair_sims[(w1, w2)]['African'] = african_sim_matrices[NUM_LAYERS][i1, i2]
    if latin_sim_matrices:
        pair_sims[(w1, w2)]['Latin'] = latin_sim_matrices[NUM_LAYERS][i1, i2]
    if offspring_sim_matrices:
        pair_sims[(w1, w2)]['Offspring'] = offspring_sim_matrices[NUM_LAYERS][i1, i2]

# Create grouped bar chart
fig = go.Figure()

pair_labels = [f"{w1} ↔ {w2}" for w1, w2 in key_pairs]
models = list(pair_sims[key_pairs[0]].keys())

for model in models:
    values = [pair_sims[pair][model] for pair in key_pairs]
    fig.add_trace(go.Bar(
        name=model,
        x=pair_labels,
        y=values,
        marker_color=MODEL_COLORS[model],
        text=[f'{v:.3f}' for v in values],
        textposition='outside',
    ))

fig.update_layout(
    title=dict(
        text="🔗 Key Word Pair Similarities Across Models",
        font=dict(size=18)
    ),
    xaxis_title="Word Pair",
    yaxis_title="Cosine Similarity",
    barmode='group',
    height=500,
    width=1000,
    template='plotly_white',
)
fig.show()
save_figure(fig, "07_key_word_pairs_all_models.html")

# Print analysis
print("\n📊 KEY FINDINGS:")
print("-" * 50)
print("\n Expected: war↔peace should be LOW, war↔destroy should be HIGH")
for pair in key_pairs:
    w1, w2 = pair
    print(f"\n   {w1} ↔ {w2}:")
    for model, sim in pair_sims[pair].items():
        print(f"      {model}: {sim:.4f}")

💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/07_key_word_pairs_all_models.html



📊 KEY FINDINGS:
--------------------------------------------------

 Expected: war↔peace should be LOW, war↔destroy should be HIGH

   war ↔ peace:
      Base: 0.4662
      African: 0.4645
      Latin: 0.4234
      Offspring: 0.4371

   war ↔ destroy:
      Base: 0.4449
      African: 0.3304
      Latin: 0.3114
      Offspring: 0.3538

   peace ↔ love:
      Base: 0.5552
      African: 0.4049
      Latin: 0.4057
      Offspring: 0.3892

   justice ↔ freedom:
      Base: 0.5549
      African: 0.4939
      Latin: 0.4086
      Offspring: 0.3850

   culture ↔ tradition:
      Base: 0.6724
      African: 0.6232
      Latin: 0.5578
      Offspring: 0.5887


In [46]:
# ============================================================================
# CELL 25: PLOT 8 - WORD TRAJECTORY ACROSS LAYERS (Selected Words)
# ============================================================================

# Track how specific word similarities change across layers
selected_pairs = [("war", "peace"), ("peace", "love")]

fig = make_subplots(
    rows=1, cols=len(selected_pairs),
    subplot_titles=[f"{w1} ↔ {w2}" for w1, w2 in selected_pairs]
)

for col, (w1, w2) in enumerate(selected_pairs, 1):
    i1 = ALL_WORDS.index(w1)
    i2 = ALL_WORDS.index(w2)

    # Base model
    layers_list = sorted(base_sim_matrices.keys())
    base_sims = [base_sim_matrices[l][i1, i2] for l in layers_list]

    fig.add_trace(go.Scatter(
        x=layers_list, y=base_sims,
        mode='lines+markers', name='Base',
        line=dict(color=MODEL_COLORS['Base'], width=2),
        showlegend=(col == 1),
    ), row=1, col=col)

    if african_sim_matrices:
        african_sims = [african_sim_matrices[l][i1, i2] for l in layers_list]
        fig.add_trace(go.Scatter(
            x=layers_list, y=african_sims,
            mode='lines+markers', name='African',
            line=dict(color=MODEL_COLORS['African'], width=2),
            showlegend=(col == 1),
        ), row=1, col=col)

    if offspring_sim_matrices:
        offspring_sims = [offspring_sim_matrices[l][i1, i2] for l in layers_list]
        fig.add_trace(go.Scatter(
            x=layers_list, y=offspring_sims,
            mode='lines+markers', name='Offspring',
            line=dict(color=MODEL_COLORS['Offspring'], width=2, dash='dash'),
            showlegend=(col == 1),
        ), row=1, col=col)

fig.update_layout(
    title=dict(
        text="📈 Word Pair Similarity Evolution Across Layers",
        font=dict(size=18)
    ),
    height=400,
    width=1000,
    template='plotly_white',
)

fig.update_xaxes(title_text="Layer")
fig.update_yaxes(title_text="Cosine Similarity")
fig.show()
save_figure(fig, "08_word_pair_evolution_layers.html")

💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/08_word_pair_evolution_layers.html


'/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/08_word_pair_evolution_layers.html'

In [47]:
# ============================================================================
# CELL 26: CRITICAL FIX - TRUE ISOLATED WORD EMBEDDING EXTRACTION
# ============================================================================

print("\n" + "=" * 70)
print("🔧 CRITICAL FIX: ISOLATED WORD EMBEDDING EXTRACTION")
print("=" * 70)

class IsolatedWordAnalyzer:
    """
    Extract word embeddings in COMPLETE ISOLATION - no context whatsoever.

    KEY INSIGHT:
    "war" and "peace" MUST have different embeddings because they have
    opposite meanings. If we process them with ANY shared context,
    the context dominates and we lose word-specific information.

    SOLUTION: Process ONLY the word itself, nothing else.
    """

    def __init__(self, device=DEVICE, eps=1e-9):
        self.device = device
        self.eps = eps

    def get_isolated_embedding(
        self,
        model,
        tokenizer,
        word: str,
        layer_idx: int
    ) -> Tuple[np.ndarray, Dict[str, float]]:
        """
        Get embedding of a SINGLE WORD processed in complete isolation.

        NO context, NO prompt template, JUST the word.
        """
        # Tokenize ONLY the word (no special tokens to avoid BOS influence)
        # But we need at least BOS for model to work, so we use it minimally

        # Method 1: Direct word encoding
        word_tokens = tokenizer.encode(word, add_special_tokens=False)

        # Create input with just BOS + word tokens
        bos_id = tokenizer.bos_token_id if tokenizer.bos_token_id else tokenizer.eos_token_id
        input_ids = torch.tensor([[bos_id] + word_tokens]).to(self.device)
        attention_mask = torch.ones_like(input_ids)

        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                return_dict=True
            )

        # Get hidden states at target layer
        if layer_idx >= len(outputs.hidden_states):
            layer_idx = len(outputs.hidden_states) - 1

        hidden = outputs.hidden_states[layer_idx].squeeze(0)  # [T, D]

        # Skip BOS token (index 0), take mean of word tokens
        if hidden.shape[0] > 1:
            word_embedding = hidden[1:].mean(dim=0)  # Average word subword tokens
        else:
            word_embedding = hidden[0]

        # Convert to numpy
        emb_np = word_embedding.detach().cpu().float().numpy()

        # Compute embedding statistics
        stats = {
            'norm': float(np.linalg.norm(emb_np)),
            'mean': float(np.mean(emb_np)),
            'std': float(np.std(emb_np)),
            'min': float(np.min(emb_np)),
            'max': float(np.max(emb_np)),
            'num_tokens': len(word_tokens),
        }

        return emb_np, stats

    def compute_cosine_similarity(self, emb1: np.ndarray, emb2: np.ndarray) -> float:
        """Compute cosine similarity between two embeddings."""
        norm1 = np.linalg.norm(emb1)
        norm2 = np.linalg.norm(emb2)
        if norm1 < 1e-10 or norm2 < 1e-10:
            return 0.0
        return float(np.dot(emb1, emb2) / (norm1 * norm2))

    def compute_euclidean_distance(self, emb1: np.ndarray, emb2: np.ndarray) -> float:
        """Compute Euclidean distance between embeddings."""
        return float(np.linalg.norm(emb1 - emb2))

    def analyze_word_across_layers(
        self,
        model,
        tokenizer,
        word: str,
        layer_indices: List[int]
    ) -> Dict[int, Dict[str, Any]]:
        """Analyze a single word across all layers."""
        results = {}

        for layer_idx in layer_indices:
            emb, stats = self.get_isolated_embedding(model, tokenizer, word, layer_idx)
            results[layer_idx] = {
                'embedding': emb,
                **stats
            }

        return results

    def analyze_all_words(
        self,
        model,
        tokenizer,
        words: List[str],
        layer_indices: List[int],
        desc: str = "Words"
    ) -> Dict[str, Dict[int, Dict]]:
        """Analyze all words across all layers."""
        all_results = {}

        for word in tqdm(words, desc=desc):
            all_results[word] = self.analyze_word_across_layers(
                model, tokenizer, word, layer_indices
            )

        return all_results

    def compute_similarity_matrix(
        self,
        word_results: Dict[str, Dict[int, Dict]],
        layer_idx: int,
        words: List[str]
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Compute pairwise similarity and distance matrices.

        Returns: (similarity_matrix, distance_matrix)
        """
        n = len(words)
        sim_matrix = np.zeros((n, n))
        dist_matrix = np.zeros((n, n))

        for i, w1 in enumerate(words):
            for j, w2 in enumerate(words):
                emb1 = word_results.get(w1, {}).get(layer_idx, {}).get('embedding')
                emb2 = word_results.get(w2, {}).get(layer_idx, {}).get('embedding')

                if emb1 is not None and emb2 is not None:
                    sim_matrix[i, j] = self.compute_cosine_similarity(emb1, emb2)
                    dist_matrix[i, j] = self.compute_euclidean_distance(emb1, emb2)

        return sim_matrix, dist_matrix


# Initialize
isolated_analyzer = IsolatedWordAnalyzer(device=DEVICE)
print("✅ Isolated Word Analyzer initialized")


🔧 CRITICAL FIX: ISOLATED WORD EMBEDDING EXTRACTION
✅ Isolated Word Analyzer initialized


In [51]:
# ============================================================================
# CELL 27: RE-ANALYZE ALL WORDS WITH ISOLATION (BASE MODEL)
# ============================================================================

print("\n" + "=" * 70)
print("📝 BASE MODEL: ISOLATED WORD ANALYSIS")
print("=" * 70)

# Reload base model if needed
try:
    _ = base_model.config
    print("Base model already loaded")
except:
    print("Reloading base model...")
    base_model, tokenizer = load_model(
        config.base_model_id,
        adapter_path=None,
        name="Base Model"
    )

# Analyze all words in isolation
base_isolated_words = isolated_analyzer.analyze_all_words(
    base_model,
    tokenizer,
    ALL_WORDS,
    ANALYSIS_LAYERS,
    desc="Base Isolated Words"
)

# Compute similarity matrices at multiple layers
sample_layers = [ANALYSIS_LAYERS[0], ANALYSIS_LAYERS[len(ANALYSIS_LAYERS)//2], ANALYSIS_LAYERS[-1]]

print(f"\n📊 Word Similarity Check at Layer {ANALYSIS_LAYERS[-1]}:")
last_layer = ANALYSIS_LAYERS[-1]

# Quick verification - war vs peace should be DIFFERENT
war_emb = base_isolated_words['war'][last_layer]['embedding']
peace_emb = base_isolated_words['peace'][last_layer]['embedding']
destroy_emb = base_isolated_words['destroy'][last_layer]['embedding']

war_peace_sim = isolated_analyzer.compute_cosine_similarity(war_emb, peace_emb)
war_destroy_sim = isolated_analyzer.compute_cosine_similarity(war_emb, destroy_emb)
peace_destroy_sim = isolated_analyzer.compute_cosine_similarity(peace_emb, destroy_emb)

print(f"   war ↔ peace:   {war_peace_sim:.4f} (should be low/negative - opposite meanings)")
print(f"   war ↔ destroy: {war_destroy_sim:.4f} (should be high - similar meanings)")
print(f"   peace ↔ destroy: {peace_destroy_sim:.4f} (should be low - opposite meanings)")

# Show embedding norms
print(f"\n📊 Word Embedding Norms at Layer {last_layer}:")
for word in ALL_WORDS[:5]: # Changed words_list to ALL_WORDS
    norm = base_isolated_words[word][last_layer]['norm']
    print(f"   {word:12s}: ||e|| = {norm:.4f}")


📝 BASE MODEL: ISOLATED WORD ANALYSIS
Base model already loaded


Base Isolated Words:   0%|          | 0/29 [00:00<?, ?it/s]


📊 Word Similarity Check at Layer 32:
   war ↔ peace:   0.4662 (should be low/negative - opposite meanings)
   war ↔ destroy: 0.4449 (should be high - similar meanings)
   peace ↔ destroy: 0.4594 (should be low - opposite meanings)

📊 Word Embedding Norms at Layer 32:
   destroy     : ||e|| = 135.7400
   war         : ||e|| = 141.3485
   protest     : ||e|| = 112.2465
   violence    : ||e|| = 115.8777
   attack      : ||e|| = 135.9059


In [54]:
# ============================================================================
# CELL 28: LOAD AFRICAN MODEL AND ANALYZE ISOLATED WORDS
# ============================================================================

print("\n" + "=" * 70)
print("📥 AFRICAN MODEL: ISOLATED WORD ANALYSIS")
print("=" * 70)

# Clear and reload
# del base_model
clear_memory()

african_isolated_words = None

if os.path.exists(config.african_adapter):
    african_model, _ = load_model(
        config.base_model_id,
        adapter_path=config.african_adapter,
        name="African Cultural Model"
    )

    african_isolated_words = isolated_analyzer.analyze_all_words(
        african_model,
        tokenizer,
        ALL_WORDS,
        ANALYSIS_LAYERS,
        desc="African Isolated Words"
    )

    # Verification
    print(f"\n📊 African Model - Word Similarity Check at Layer {last_layer}:")
    war_emb_af = african_isolated_words['war'][last_layer]['embedding']
    peace_emb_af = african_isolated_words['peace'][last_layer]['embedding']

    war_peace_sim_af = isolated_analyzer.compute_cosine_similarity(war_emb_af, peace_emb_af)
    print(f"   war ↔ peace: {war_peace_sim_af:.4f}")

    del african_model
    clear_memory()
else:
    print(f"⚠️ African adapter not found: {config.african_adapter}")


📥 AFRICAN MODEL: ISOLATED WORD ANALYSIS

📥 Loading African Cultural Model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

   Loading adapter: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/africa_adapter
   ✅ Adapter merged successfully
   ✅ African Cultural Model: 32 layers, 4096d embeddings


African Isolated Words:   0%|          | 0/29 [00:00<?, ?it/s]


📊 African Model - Word Similarity Check at Layer 32:
   war ↔ peace: 0.4648


In [56]:
# ============================================================================
# CELL 29: LOAD LATIN MODEL AND ANALYZE ISOLATED WORDS
# ============================================================================

print("\n" + "=" * 70)
print("📥 LATIN AMERICAN MODEL: ISOLATED WORD ANALYSIS")
print("=" * 70)

latin_isolated_words = None

if os.path.exists(config.latin_adapter):
    latin_model, _ = load_model(
        config.base_model_id,
        adapter_path=config.latin_adapter,
        name="Latin American Cultural Model"
    )

    latin_isolated_words = isolated_analyzer.analyze_all_words(
        latin_model,
        tokenizer,
        ALL_WORDS,
        ANALYSIS_LAYERS,
        desc="Latin Isolated Words"
    )

    print(f"\n📊 Latin Model - Word Similarity Check at Layer {last_layer}:")
    war_emb_lt = latin_isolated_words['war'][last_layer]['embedding']
    peace_emb_lt = latin_isolated_words['peace'][last_layer]['embedding']

    war_peace_sim_lt = isolated_analyzer.compute_cosine_similarity(war_emb_lt, peace_emb_lt)
    print(f"   war ↔ peace: {war_peace_sim_lt:.4f}")

    del latin_model
    clear_memory()
else:
    print(f"⚠️ Latin adapter not found: {config.latin_adapter}")


📥 LATIN AMERICAN MODEL: ISOLATED WORD ANALYSIS

📥 Loading Latin American Cultural Model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

   Loading adapter: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/latin_adapter
   ✅ Adapter merged successfully
   ✅ Latin American Cultural Model: 32 layers, 4096d embeddings


Latin Isolated Words:   0%|          | 0/29 [00:00<?, ?it/s]


📊 Latin Model - Word Similarity Check at Layer 32:
   war ↔ peace: 0.4234


In [57]:
# ============================================================================
# CELL 30: OFFSPRING MODEL - ISOLATED WORD ANALYSIS
# ============================================================================

print("\n" + "=" * 70)
print("📥 OFFSPRING MODEL: ISOLATED WORD ANALYSIS")
print("=" * 70)

offspring_isolated_words = None

if os.path.exists(config.merged_output):
    offspring_model, _ = load_model(
        config.merged_output,
        adapter_path=None,
        name="Offspring Model"
    )

    offspring_isolated_words = isolated_analyzer.analyze_all_words(
        offspring_model,
        tokenizer,
        ALL_WORDS,
        ANALYSIS_LAYERS,
        desc="Offspring Isolated Words"
    )

    print(f"\n📊 Offspring Model - Word Similarity Check at Layer {last_layer}:")
    war_emb_off = offspring_isolated_words['war'][last_layer]['embedding']
    peace_emb_off = offspring_isolated_words['peace'][last_layer]['embedding']

    war_peace_sim_off = isolated_analyzer.compute_cosine_similarity(war_emb_off, peace_emb_off)
    print(f"   war ↔ peace: {war_peace_sim_off:.4f}")

    del offspring_model
    clear_memory()
else:
    print(f"⚠️ Offspring model not found: {config.merged_output}")
    print("   Run the Fisher merge cell first!")


📥 OFFSPRING MODEL: ISOLATED WORD ANALYSIS

📥 Loading Offspring Model...


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.
The tokenizer you are loading from '/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/merged_offspring_model/' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


   ✅ Offspring Model: 32 layers, 4096d embeddings


Offspring Isolated Words:   0%|          | 0/29 [00:00<?, ?it/s]


📊 Offspring Model - Word Similarity Check at Layer 32:
   war ↔ peace: 0.4371


In [58]:
# ============================================================================
# CELL 31: RELOAD BASE MODEL FOR COMPARISONS
# ============================================================================

print("\n📥 Reloading base model for visualization comparisons...")
base_model, tokenizer = load_model(
    config.base_model_id,
    adapter_path=None,
    name="Base Model (for comparisons)"
)

# Re-extract base isolated words if not available
if 'base_isolated_words' not in dir() or base_isolated_words is None:
    base_isolated_words = isolated_analyzer.analyze_all_words(
        base_model,
        tokenizer,
        ALL_WORDS,
        ANALYSIS_LAYERS,
        desc="Base Isolated Words"
    )

print("✅ Ready for visualizations")


📥 Reloading base model for visualization comparisons...

📥 Loading Base Model (for comparisons)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

   ✅ Base Model (for comparisons): 32 layers, 4096d embeddings
✅ Ready for visualizations


In [62]:
# ============================================================================
# CELL 32: PLOT 9 - WORD SIMILARITY HEATMAP (FIXED - ISOLATED)
# ============================================================================

print("\n📊 Generating Word Similarity Heatmaps...")

# Compute similarity matrix at final layer
sim_matrix_base_isolated, dist_matrix_base = isolated_analyzer.compute_similarity_matrix(
    base_isolated_words, ANALYSIS_LAYERS[-1], ALL_WORDS
)

# Create attractive heatmap with annotations
fig = go.Figure()

# Custom colorscale: Red (dissimilar) -> White (neutral) -> Blue (similar)
colorscale = [
    [0.0, '#d73027'],   # Dark red (dissimilar, -1)
    [0.25, '#fc8d59'],  # Orange-red
    [0.5, '#ffffbf'],   # Yellow/white (neutral, 0)
    [0.75, '#91bfdb'],  # Light blue
    [1.0, '#4575b4']    # Dark blue (similar, +1)
]

fig.add_trace(go.Heatmap(
    z=sim_matrix_base_isolated,
    x=[w.capitalize() for w in ALL_WORDS],
    y=[w.capitalize() for w in ALL_WORDS],
    colorscale=colorscale,
    zmin=-0.5,
    zmax=1.0,
    colorbar=dict(
        title="Cosine<br>Similarity",
        titleside="right",
        tickvals=[-0.5, 0, 0.5, 1.0],
        ticktext=["-0.5 (opposite)", "0", "0.5", "1.0 (identical)"]
    ),
    text=np.round(sim_matrix_base_isolated, 2),
    texttemplate="%{text}",
    textfont={"size": 9, "color": "black"},
    hovertemplate="<b>%{y}</b> ↔ <b>%{x}</b><br>Similarity: %{z:.3f}<extra></extra>"
))

# Add category boxes
fig.add_shape(type="rect", x0=-0.5, y0=-0.5, x1=2.5, y1=2.5,
              line=dict(color="red", width=3), fillcolor="rgba(0,0,0,0)")
fig.add_shape(type="rect", x0=2.5, y0=2.5, x1=6.5, y1=6.5,
              line=dict(color="green", width=3), fillcolor="rgba(0,0,0,0)")
fig.add_shape(type="rect", x0=6.5, y0=6.5, x1=14.5, y1=14.5,
              line=dict(color="blue", width=3), fillcolor="rgba(0,0,0,0)")

fig.add_annotation(x=1, y=15.5, text="NEGATIVE", showarrow=False,
                   font=dict(color="red", size=12, weight="bold"))
fig.add_annotation(x=4.5, y=15.5, text="POSITIVE", showarrow=False,
                   font=dict(color="green", size=12, weight="bold"))
fig.add_annotation(x=10.5, y=15.5, text="NEUTRAL", showarrow=False,
                   font=dict(color="blue", size=12, weight="bold"))

fig.update_layout(
    title=dict(
        text=f"Word Similarity Matrix (Isolated Embeddings) - Layer {ANALYSIS_LAYERS[-1]}<br><sup>Base Model | Red=Dissimilar, Blue=Similar</sup>",
        font=dict(size=16)
    ),
    height=750, width=850,
    template='plotly_white',
    xaxis=dict(tickangle=45, side='bottom'),
    yaxis=dict(autorange='reversed'),
)
fig.show()
save_figure(fig, "09_word_similarity_heatmap_base.html")

# Save as CSV
sim_df = pd.DataFrame(sim_matrix_base_isolated,
                      index=[w.capitalize() for w in ALL_WORDS],
                      columns=[w.capitalize() for w in ALL_WORDS])
sim_df.to_csv(os.path.join(config.output_dir, "word_similarity_matrix_base.csv"))
print(f"💾 Saved: word_similarity_matrix_base.csv")


📊 Generating Word Similarity Heatmaps...


💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/09_word_similarity_heatmap_base.html


💾 Saved: word_similarity_matrix_base.csv


In [66]:
# # ============================================================================
# # CELL 33: PLOT 10 - WORD EMBEDDING NORM TRAJECTORIES
# # ============================================================================

# print("\n📊 Generating Word Embedding Norm Trajectories...")

# fig = make_subplots(
#     rows=1, cols=3,
#     subplot_titles=[
#         'Negative Words (destroy, war, protest)',
#         'Positive Words (peace, justice, freedom, wisdom)',
#         'Neutral Words (skill, concept, foundation, ...)'
#     ],
#     horizontal_spacing=0.08
# )

# # Plot by category
# for col, (cat_name, cat_words) in enumerate(WORD_CATEGORIES.items(), 1):
#     for word in cat_words:
#         if word not in base_isolated_words:
#             continue

#         layers = sorted(base_isolated_words[word].keys())
#         norms = [base_isolated_words[word][l]['norm'] for l in layers]

#         fig.add_trace(go.Scatter(
#             x=layers, y=norms,
#             mode='lines+markers',
#             name=word.capitalize(),
#             line=dict(color=WORD_CATEGORIES["abstract"]["words"].get(word, '#666'), width=2),
#             marker=dict(size=4),
#             legendgroup=cat_name,
#             showlegend=True
#         ), row=1, col=col)

# fig.update_xaxes(title_text="Layer", row=1, col=2)
# fig.update_yaxes(title_text="Embedding Norm ||e||", row=1, col=1)

# fig.update_layout(
#     title=dict(text="Word Embedding Norm by Layer (Isolated) - Base Model", font=dict(size=16)),
#     height=450, width=1200,
#     template='plotly_white',
#     legend=dict(x=1.02, y=0.98, font=dict(size=9))
# )
# fig.show()
# save_figure(fig, "10_word_embedding_norms_by_category.html")


📊 Generating Word Embedding Norm Trajectories...


💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/10_word_embedding_norms_by_category.html


'/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/10_word_embedding_norms_by_category.html'

In [69]:
# ============================================================================
# CELL 34: PLOT 11 - SEMANTIC OPPOSITES DISTANCE TRAJECTORY
# ============================================================================

print("\n📊 Generating Semantic Opposites Distance Trajectory...")

# Define opposite pairs
opposite_pairs = [
    ('war', 'peace'),
    ('destroy', 'wisdom'),
    ('protest', 'order'),
]

# Define similar pairs
similar_pairs = [
    ('war', 'destroy'),
    ('peace', 'freedom'),
    ('belief', 'tradition'),
]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Opposite Meaning Pairs (Should be DISTANT)',
                    'Similar Meaning Pairs (Should be CLOSE)']
)

# Opposite pairs
for w1, w2 in opposite_pairs:
    if w1 not in base_isolated_words or w2 not in base_isolated_words:
        continue

    layers = sorted(base_isolated_words[w1].keys())
    similarities = []
    for l in layers:
        emb1 = base_isolated_words[w1][l]['embedding']
        emb2 = base_isolated_words[w2][l]['embedding']
        sim = isolated_analyzer.compute_cosine_similarity(emb1, emb2)
        similarities.append(sim)

    fig.add_trace(go.Scatter(
        x=layers, y=similarities,
        mode='lines+markers',
        name=f"{w1} ↔ {w2}",
        line=dict(width=2),
        marker=dict(size=5)
    ), row=1, col=1)

# Similar pairs
for w1, w2 in similar_pairs:
    if w1 not in base_isolated_words or w2 not in base_isolated_words:
        continue

    layers = sorted(base_isolated_words[w1].keys())
    similarities = []
    for l in layers:
        emb1 = base_isolated_words[w1][l]['embedding']
        emb2 = base_isolated_words[w2][l]['embedding']
        sim = isolated_analyzer.compute_cosine_similarity(emb1, emb2)
        similarities.append(sim)

    fig.add_trace(go.Scatter(
        x=layers, y=similarities,
        mode='lines+markers',
        name=f"{w1} ↔ {w2}",
        line=dict(width=2),
        marker=dict(size=5)
    ), row=1, col=2)

# Add reference lines
fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=2)

fig.update_xaxes(title_text="Layer")
fig.update_yaxes(title_text="Cosine Similarity", row=1, col=1)

fig.update_layout(
    title=dict(text="Semantic Pair Similarity by Layer - Base Model", font=dict(size=16)),
    height=450, width=1100,
    template='plotly_white'
)
fig.show()
save_figure(fig, "11_semantic_pair_similarity.html")


📊 Generating Semantic Opposites Distance Trajectory...


💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/11_semantic_pair_similarity.html


'/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/11_semantic_pair_similarity.html'

In [72]:
# ============================================================================
# CELL 35: PLOT 12 - MODEL COMPARISON: BASE VS AFRICAN (WORDS)
# ============================================================================

print("\n📊 Generating Model Comparison: Base vs African...")

if african_isolated_words is not None:

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            'war ↔ peace Similarity',
            'destroy ↔ wisdom Similarity',
            'Embedding Norm: "culture"',
            'Embedding Norm: "tradition"'
        ],
        vertical_spacing=0.12
    )

    layers = sorted(base_isolated_words['war'].keys())

    # War-Peace similarity comparison
    base_war_peace = [isolated_analyzer.compute_cosine_similarity(
        base_isolated_words['war'][l]['embedding'],
        base_isolated_words['peace'][l]['embedding']
    ) for l in layers]

    african_war_peace = [isolated_analyzer.compute_cosine_similarity(
        african_isolated_words['war'][l]['embedding'],
        african_isolated_words['peace'][l]['embedding']
    ) for l in layers]

    fig.add_trace(go.Scatter(x=layers, y=base_war_peace, name='Base',
                             line=dict(color='#666', width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=layers, y=african_war_peace, name='African',
                             line=dict(color='#234', width=2)), row=1, col=1)

    # Destroy-Wisdom similarity
    base_destroy_wisdom = [isolated_analyzer.compute_cosine_similarity(
        base_isolated_words['destroy'][l]['embedding'],
        base_isolated_words['wisdom'][l]['embedding']
    ) for l in layers]

    african_destroy_wisdom = [isolated_analyzer.compute_cosine_similarity(
        african_isolated_words['destroy'][l]['embedding'],
        african_isolated_words['wisdom'][l]['embedding']
    ) for l in layers]

    fig.add_trace(go.Scatter(x=layers, y=base_destroy_wisdom, name='Base',
                             line=dict(color='#666', width=2), showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=layers, y=african_destroy_wisdom, name='African',
                             line=dict(color='#234', width=2), showlegend=False), row=1, col=2)

    # Culture embedding norm
    base_culture_norm = [base_isolated_words['culture'][l]['norm'] for l in layers]
    african_culture_norm = [african_isolated_words['culture'][l]['norm'] for l in layers]

    fig.add_trace(go.Scatter(x=layers, y=base_culture_norm, name='Base',
                             line=dict(color='#666', width=2), showlegend=False), row=2, col=1)
    fig.add_trace(go.Scatter(x=layers, y=african_culture_norm, name='African',
                             line=dict(color='#234', width=2), showlegend=False), row=2, col=1)

    # Tradition embedding norm
    base_tradition_norm = [base_isolated_words['tradition'][l]['norm'] for l in layers]
    african_tradition_norm = [african_isolated_words['tradition'][l]['norm'] for l in layers]

    fig.add_trace(go.Scatter(x=layers, y=base_tradition_norm, name='Base',
                             line=dict(color='#666', width=2), showlegend=False), row=2, col=2)
    fig.add_trace(go.Scatter(x=layers, y=african_tradition_norm, name='African',
                             line=dict(color='#234', width=2), showlegend=False), row=2, col=2)

    fig.update_layout(
        title=dict(text="Word Representation: Base vs African Model", font=dict(size=16)),
        height=600, width=1000,
        template='plotly_white'
    )
    fig.show()
    save_figure(fig, "12_base_vs_african_words.html")
else:
    print("⚠️ Skipping - African model results not available")


📊 Generating Model Comparison: Base vs African...


💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/12_base_vs_african_words.html


In [79]:
# ============================================================================
# CELL 36: PLOT 13 - CULTURAL WORD DISTANCE FROM BASE MODEL
# ============================================================================

print("\n📊 Generating Cultural Word Distance from Base...")

def compute_word_distance_from_base(
    base_results: Dict,
    model_results: Dict,
    layer_idx: int,
    words: List[str]
) -> Dict[str, float]:
    """Compute how far each word embedding moved from base model."""
    distances = {}

    for word in words:
        if word not in base_results or word not in model_results:
            continue

        base_emb = base_results[word][layer_idx]['embedding']
        model_emb = model_results[word][layer_idx]['embedding']

        # Cosine distance = 1 - cosine_similarity
        cos_sim = isolated_analyzer.compute_cosine_similarity(base_emb, model_emb)
        cos_dist = 1 - cos_sim

        distances[word] = cos_dist

    return distances

# Compute distances at final layer
models_to_compare = []
if african_isolated_words:
    models_to_compare.append(('African', african_isolated_words, MODEL_COLORS['African']))
if latin_isolated_words:
    models_to_compare.append(('Latin', latin_isolated_words, MODEL_COLORS['Latin']))
if offspring_isolated_words:
    models_to_compare.append(('Offspring', offspring_isolated_words, MODEL_COLORS['Offspring']))

if models_to_compare:
    fig = go.Figure()

    for model_name, model_results, color in models_to_compare:
        distances = compute_word_distance_from_base(
            base_isolated_words, model_results, ANALYSIS_LAYERS[-1], ALL_WORDS
        )

        words_sorted = sorted(distances.keys(), key=lambda w: distances[w], reverse=True)

        fig.add_trace(go.Bar(
            x=[w.capitalize() for w in words_sorted],
            y=[distances[w] for w in words_sorted],
            name=model_name,
            marker_color=color,
        ))

    fig.update_layout(
        title=dict(text=f"Word Distance from Base Model (Layer {ANALYSIS_LAYERS[-1]})<br><sup>Higher = More Cultural Shift</sup>",
                   font=dict(size=16)),
        xaxis_title="Word",
        yaxis_title="Cosine Distance from Base",
        barmode='group',
        height=500, width=1100,
        template='plotly_white',
        xaxis=dict(tickangle=45)
    )
    fig.show()
    save_figure(fig, "13_cultural_word_distance_from_base.html")

    # Save as CSV
    dist_data = []
    for model_name, model_results, _ in models_to_compare:
        distances = compute_word_distance_from_base(
            base_isolated_words, model_results, ANALYSIS_LAYERS[-1], ALL_WORDS
        )
        for word, dist in distances.items():
            dist_data.append({'Model': model_name, 'Word': word, 'Distance': dist})

    dist_df = pd.DataFrame(dist_data)
    dist_df.to_csv(os.path.join(config.output_dir, "cultural_word_distances.csv"), index=False)
    print(f"💾 Saved: cultural_word_distances.csv")
else:
    print("⚠️ No fine-tuned models available for comparison")


📊 Generating Cultural Word Distance from Base...


💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/13_cultural_word_distance_from_base.html


💾 Saved: cultural_word_distances.csv


In [81]:
# ============================================================================
# CELL 37: PLOT 14 - 3D WORD EMBEDDING SPACE (PCA)
# ============================================================================

print("\n📊 Generating 3D Word Embedding Space...")

from sklearn.decomposition import PCA

# Get embeddings at final layer for all words
embeddings_base = np.array([
    base_isolated_words[w][ANALYSIS_LAYERS[-1]]['embedding']
    for w in ALL_WORDS
])

# PCA to 3D
pca = PCA(n_components=3)
embeddings_3d = pca.fit_transform(embeddings_base)

# Create 3D scatter
fig = go.Figure()

# Color by category
for cat_name, cat_info in WORD_CATEGORIES.items():
    # Use the color directly from the WORD_CATEGORIES dictionary
    color = cat_info['color']

    indices = [ALL_WORDS.index(w) for w in cat_info['words'] if w in ALL_WORDS]

    fig.add_trace(go.Scatter3d(
        x=embeddings_3d[indices, 0],
        y=embeddings_3d[indices, 1],
        z=embeddings_3d[indices, 2],
        mode='markers+text',
        name=cat_name.capitalize(),
        marker=dict(size=8, color=color, opacity=0.8),
        text=[ALL_WORDS[i].capitalize() for i in indices],
        textposition='top center',
        textfont=dict(size=10)
    ))

# Add lines between opposite pairs
for w1, w2 in opposite_pairs:
    if w1 in ALL_WORDS and w2 in ALL_WORDS:
        i1, i2 = ALL_WORDS.index(w1), ALL_WORDS.index(w2)
        fig.add_trace(go.Scatter3d(
            x=[embeddings_3d[i1, 0], embeddings_3d[i2, 0]],
            y=[embeddings_3d[i1, 1], embeddings_3d[i2, 1]],
            z=[embeddings_3d[i1, 2], embeddings_3d[i2, 2]],
            mode='lines',
            line=dict(color='red', width=2, dash='dash'),
            showlegend=False,
            hoverinfo='skip'
        ))

fig.update_layout(
    title=dict(text=f"3D Word Embedding Space (PCA) - Layer {ANALYSIS_LAYERS[-1]}<br><sup>Base Model | Dashed lines = Opposite pairs</sup>",
               font=dict(size=16)),
    scene=dict(
        xaxis_title=f"PC1 ({pca.explained_variance_ratio_[0]:.1%})",
        yaxis_title=f"PC2 ({pca.explained_variance_ratio_[1]:.1%})",
        zaxis_title=f"PC3 ({pca.explained_variance_ratio_[2]:.1%})"
    ),
    height=700, width=900,
    template='plotly_white'
)
fig.show()
save_figure(fig, "14_word_embedding_3d_pca.html")


📊 Generating 3D Word Embedding Space...


💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/14_word_embedding_3d_pca.html


'/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/14_word_embedding_3d_pca.html'

In [87]:
# ============================================================================
# CELL 38: PLOT 15 - MODEL MERGE VALIDATION: OFFSPRING INTERPOLATION
# ============================================================================

print("\n📊 Generating Model Merge Validation...")

if offspring_isolated_words and african_isolated_words and latin_isolated_words:

    # For each word, check if offspring is between parents
    fig = go.Figure()

    validation_results = []

    for word in ALL_WORDS:
        # Get embeddings
        base_emb = base_isolated_words[word][ANALYSIS_LAYERS[-1]]['embedding']
        african_emb = african_isolated_words[word][ANALYSIS_LAYERS[-1]]['embedding']
        latin_emb = latin_isolated_words[word][ANALYSIS_LAYERS[-1]]['embedding']
        offspring_emb = offspring_isolated_words[word][ANALYSIS_LAYERS[-1]]['embedding']

        # Compute distances
        offspring_to_base = isolated_analyzer.compute_cosine_similarity(offspring_emb, base_emb)
        offspring_to_african = isolated_analyzer.compute_cosine_similarity(offspring_emb, african_emb)
        offspring_to_latin = isolated_analyzer.compute_cosine_similarity(offspring_emb, latin_emb)
        african_to_latin = isolated_analyzer.compute_cosine_similarity(african_emb, latin_emb)

        # Check interpolation: offspring should be similar to both parents
        avg_parent_sim = (offspring_to_african + offspring_to_latin) / 2

        validation_results.append({
            'word': word,
            'offspring_to_base': offspring_to_base,
            'offspring_to_african': offspring_to_african,
            'offspring_to_latin': offspring_to_latin,
            'african_to_latin': african_to_latin,
            'avg_parent_sim': avg_parent_sim,
            'is_interpolated': offspring_to_african > 0.9 and offspring_to_latin > 0.9
        })

    # Create grouped bar chart
    words_sorted = sorted(validation_results, key=lambda x: x['avg_parent_sim'], reverse=True)

    fig.add_trace(go.Bar(
        x=[r['word'].capitalize() for r in words_sorted],
        y=[r['offspring_to_african'] for r in words_sorted],
        name='Offspring ↔ African',
        marker_color='red'
    ))

    fig.add_trace(go.Bar(
        x=[r['word'].capitalize() for r in words_sorted],
        y=[r['offspring_to_latin'] for r in words_sorted],
        name='Offspring ↔ Latin',
        marker_color='green'
    ))

    fig.add_trace(go.Bar(
        x=[r['word'].capitalize() for r in words_sorted],
        y=[r['african_to_latin'] for r in words_sorted],
        name='African ↔ Latin',
        marker_color='blue'
    ))

    fig.add_hline(y=0.95, line_dash="dash", line_color="green",
                  annotation_text="High similarity threshold")

    fig.update_layout(
        title=dict(text="Fisher Merge Validation: Offspring Parent Similarity<br><sup>Higher bars = Offspring inherited from parent</sup>",
                   font=dict(size=16)),
        xaxis_title="Word",
        yaxis_title="Cosine Similarity",
        barmode='group',
        height=500, width=1100,
        template='plotly_white',
        xaxis=dict(tickangle=45)
    )
    fig.show()
    save_figure(fig, "15_merge_validation_offspring_parents.html")

    # Validation summary
    valid_count = sum(1 for r in validation_results if r['is_interpolated'])
    print(f"\n✅ MERGE VALIDATION SUMMARY:")
    print(f"   Words with valid interpolation (>0.9 sim to both parents): {valid_count}/{len(validation_results)}")

    # Save validation results
    val_df = pd.DataFrame(validation_results)
    val_df.to_csv(os.path.join(config.output_dir, "merge_validation_results.csv"), index=False)
    print(f"💾 Saved: merge_validation_results.csv")

else:
    print("⚠️ Skipping merge validation - not all models available")


📊 Generating Model Merge Validation...


💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/15_merge_validation_offspring_parents.html



✅ MERGE VALIDATION SUMMARY:
   Words with valid interpolation (>0.9 sim to both parents): 22/29
💾 Saved: merge_validation_results.csv


In [88]:
# ============================================================================
# CELL 39: PLOT 16 - ALL MODELS SIMILARITY HEATMAPS (SIDE BY SIDE)
# ============================================================================

print("\n📊 Generating All Models Similarity Heatmaps...")

available_models = [('Base', base_isolated_words)]
if african_isolated_words:
    available_models.append(('African', african_isolated_words))
if latin_isolated_words:
    available_models.append(('Latin', latin_isolated_words))
if offspring_isolated_words:
    available_models.append(('Offspring', offspring_isolated_words))

num_models = len(available_models)

fig = make_subplots(
    rows=1, cols=num_models,
    subplot_titles=[name for name, _ in available_models],
    horizontal_spacing=0.05
)

colorscale = [
    [0.0, '#d73027'],
    [0.25, '#fc8d59'],
    [0.5, '#ffffbf'],
    [0.75, '#91bfdb'],
    [1.0, '#4575b4']
]

for col, (model_name, model_results) in enumerate(available_models, 1):
    sim_matrix, _ = isolated_analyzer.compute_similarity_matrix(
        model_results, ANALYSIS_LAYERS[-1], ALL_WORDS
    )

    fig.add_trace(go.Heatmap(
        z=sim_matrix,
        x=[w[:5] for w in ALL_WORDS],  # Shortened names
        y=[w[:5] for w in ALL_WORDS],
        colorscale=colorscale,
        zmin=-0.3,
        zmax=1.0,
        showscale=(col == num_models),
        colorbar=dict(title="Sim", x=1.02) if col == num_models else None,
    ), row=1, col=col)

fig.update_layout(
    title=dict(text=f"Word Similarity Matrices Across Models (Layer {ANALYSIS_LAYERS[-1]})",
               font=dict(size=16)),
    height=500, width=300 * num_models,
    template='plotly_white'
)

for i in range(1, num_models + 1):
    fig.update_xaxes(tickangle=45, row=1, col=i)
    fig.update_yaxes(autorange='reversed', row=1, col=i)
fig.show()
save_figure(fig, "16_all_models_similarity_heatmaps.html")


📊 Generating All Models Similarity Heatmaps...


💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/16_all_models_similarity_heatmaps.html


'/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/16_all_models_similarity_heatmaps.html'

In [90]:
# ============================================================================
# CELL 40: PLOT 17 - LAYER-WISE WORD EVOLUTION (ANIMATED)
# ============================================================================

print("\n📊 Generating Layer-wise Word Evolution...")

# Track how word pairs similarity evolves across layers
fig = go.Figure()

word_pairs_to_track = [
    ('war', 'peace', 'Opposites'),
    ('war', 'destroy', 'Similar'),
    ('culture', 'tradition', 'Similar')
]

colors = ['#E63946', '#2A9D8F', '#F4A261', '#7209B7']

for (w1, w2, label), color in zip(word_pairs_to_track, colors):
    layers = sorted(base_isolated_words[w1].keys())
    sims = []

    for l in layers:
        emb1 = base_isolated_words[w1][l]['embedding']
        emb2 = base_isolated_words[w2][l]['embedding']
        sims.append(isolated_analyzer.compute_cosine_similarity(emb1, emb2))

    fig.add_trace(go.Scatter(
        x=layers, y=sims,
        mode='lines+markers',
        name=f"{w1} ↔ {w2} ({label})",
        line=dict(color=color, width=3),
        marker=dict(size=6)
    ))

fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)

fig.update_layout(
    title=dict(text="Word Pair Similarity Evolution Across Layers - Base Model", font=dict(size=16)),
    xaxis_title="Layer",
    yaxis_title="Cosine Similarity",
    height=500, width=1000,
    template='plotly_white',
    legend=dict(x=0.01, y=0.99)
)
fig.show()
save_figure(fig, "17_word_pair_evolution.html")


📊 Generating Layer-wise Word Evolution...


💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/17_word_pair_evolution.html


'/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/17_word_pair_evolution.html'

In [91]:
# ============================================================================
# CELL 41: PLOT 18 - CULTURAL SHIFT RADAR CHART
# ============================================================================

print("\n📊 Generating Cultural Shift Radar Chart...")

if african_isolated_words or latin_isolated_words:

    # Select key cultural words
    cultural_words = ['culture', 'tradition', 'belief', 'wisdom', 'justice',
                      'freedom', 'peace', 'order', 'protest', 'war']

    fig = go.Figure()

    # For each model, compute distance from base for each cultural word
    for model_name, model_results, color in models_to_compare:
        distances = []
        for word in cultural_words:
            if word in base_isolated_words and word in model_results:
                base_emb = base_isolated_words[word][ANALYSIS_LAYERS[-1]]['embedding']
                model_emb = model_results[word][ANALYSIS_LAYERS[-1]]['embedding']
                dist = 1 - isolated_analyzer.compute_cosine_similarity(base_emb, model_emb)
                distances.append(dist)
            else:
                distances.append(0)

        # Close the radar
        distances.append(distances[0])
        words_radar = cultural_words + [cultural_words[0]]

        fig.add_trace(go.Scatterpolar(
            r=distances,
            theta=[w.capitalize() for w in words_radar],
            name=model_name,
            fill='toself',
            fillcolor=color.replace(')', ', 0.2)').replace('rgb', 'rgba'),
            line=dict(color=color, width=2)
        ))

    fig.update_layout(
        title=dict(text="Cultural Word Shift from Base Model<br><sup>Larger area = More cultural adaptation</sup>",
                   font=dict(size=16)),
        polar=dict(
            radialaxis=dict(visible=True, range=[0, 0.3])
        ),
        height=600, width=700,
        template='plotly_white'
    )
    fig.show()
    save_figure(fig, "18_cultural_shift_radar.html")
else:
    print("⚠️ No cultural models available for radar chart")


📊 Generating Cultural Shift Radar Chart...


💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/18_cultural_shift_radar.html


In [93]:
# ============================================================================
# CELL 42: COMPREHENSIVE RESULTS TABLE
# ============================================================================

print("\n📊 Generating Comprehensive Results Tables...")

# Table 1: Word Metrics at Final Layer
word_metrics = []
for word in ALL_WORDS:
    row = {
        'Word': word.capitalize(),
        'Category': [k for k, v in WORD_CATEGORIES.items() if word in v['words']][0],
        'Base_Norm': base_isolated_words[word][ANALYSIS_LAYERS[-1]]['norm'],
        'Base_Mean': base_isolated_words[word][ANALYSIS_LAYERS[-1]]['mean'],
        'Base_Std': base_isolated_words[word][ANALYSIS_LAYERS[-1]]['std'],
    }

    if african_isolated_words and word in african_isolated_words:
        row['African_Norm'] = african_isolated_words[word][ANALYSIS_LAYERS[-1]]['norm']
        base_emb = base_isolated_words[word][ANALYSIS_LAYERS[-1]]['embedding']
        af_emb = african_isolated_words[word][ANALYSIS_LAYERS[-1]]['embedding']
        row['African_Shift'] = 1 - isolated_analyzer.compute_cosine_similarity(base_emb, af_emb)

    if latin_isolated_words and word in latin_isolated_words:
        row['Latin_Norm'] = latin_isolated_words[word][ANALYSIS_LAYERS[-1]]['norm']
        base_emb = base_isolated_words[word][ANALYSIS_LAYERS[-1]]['embedding']
        lt_emb = latin_isolated_words[word][ANALYSIS_LAYERS[-1]]['embedding']
        row['Latin_Shift'] = 1 - isolated_analyzer.compute_cosine_similarity(base_emb, lt_emb)

    word_metrics.append(row)

metrics_df = pd.DataFrame(word_metrics)
metrics_df = metrics_df.sort_values('Category')

print("\n📋 WORD METRICS SUMMARY:")
print(metrics_df.to_string(index=False))

metrics_df.to_csv(os.path.join(config.output_dir, "word_metrics_summary.csv"), index=False)
print(f"\n💾 Saved: word_metrics_summary.csv")


📊 Generating Comprehensive Results Tables...

📋 WORD METRICS SUMMARY:
       Word Category  Base_Norm  Base_Mean  Base_Std  African_Norm  African_Shift  Latin_Norm  Latin_Shift
 Understand abstract 119.460144   0.066541  1.865378    119.804047       0.339601  118.746452     0.333947
   Hardwork abstract 116.561684   0.028791  1.821049    111.237114       0.281913  110.786591     0.330295
      Skill abstract 139.515701   0.047829  2.179408    142.559479       0.466105  144.530945     0.488574
      Logic abstract 139.900238   0.042888  2.185520    142.857346       0.494446  143.274185     0.448472
     Reason abstract 133.802750   0.067477  2.089579    138.725113       0.576328  140.912750     0.487152
    Thought abstract 141.063614   0.020598  2.204023    141.770859       0.460928  142.968750     0.452737
       Idea abstract 141.122818   0.042138  2.204641    140.381760       0.590357  143.492371     0.534719
    Concept abstract 134.740326   0.007787  2.105303    138.924911       

In [94]:
# ============================================================================
# CELL 43: WORD-TO-MODEL AFFINITY ANALYSIS
# ============================================================================

print("\n📊 Generating Word-to-Model Affinity Analysis...")

if african_isolated_words and latin_isolated_words:

    # For each word, which cultural model is it closer to?
    affinity_data = []

    for word in ALL_WORDS:
        base_emb = base_isolated_words[word][ANALYSIS_LAYERS[-1]]['embedding']
        african_emb = african_isolated_words[word][ANALYSIS_LAYERS[-1]]['embedding']
        latin_emb = latin_isolated_words[word][ANALYSIS_LAYERS[-1]]['embedding']

        # Compute shifts from base
        african_shift = 1 - isolated_analyzer.compute_cosine_similarity(base_emb, african_emb)
        latin_shift = 1 - isolated_analyzer.compute_cosine_similarity(base_emb, latin_emb)

        # Relative affinity: positive = more African, negative = more Latin
        affinity = african_shift - latin_shift

        affinity_data.append({
            'word': word,
            'african_shift': african_shift,
            'latin_shift': latin_shift,
            'affinity': affinity,  # positive = African, negative = Latin
            'dominant': 'African' if affinity > 0.01 else ('Latin' if affinity < -0.01 else 'Balanced')
        })

    # Sort by affinity
    affinity_sorted = sorted(affinity_data, key=lambda x: x['affinity'], reverse=True)

    fig = go.Figure()

    colors = ['#F18F01' if a['affinity'] > 0.01 else ('#7B2D8E' if a['affinity'] < -0.01 else '#888888')
              for a in affinity_sorted]

    fig.add_trace(go.Bar(
        x=[a['word'].capitalize() for a in affinity_sorted],
        y=[a['affinity'] for a in affinity_sorted],
        marker_color=colors,
        hovertemplate="<b>%{x}</b><br>Affinity: %{y:.4f}<br><extra></extra>"
    ))

    fig.add_hline(y=0, line_color="black", line_width=2)

    fig.add_annotation(x=0.1, y=0.15, text="← African Affinity",
                       xref="paper", yref="y", showarrow=False, font=dict(color='#F18F01', size=12))
    fig.add_annotation(x=0.9, y=-0.15, text="Latin Affinity →",
                       xref="paper", yref="y", showarrow=False, font=dict(color='#7B2D8E', size=12))

    fig.update_layout(
        title=dict(text="Word Cultural Affinity: African vs Latin Model<br><sup>Positive = More shifted in African model | Negative = More shifted in Latin model</sup>",
                   font=dict(size=16)),
        xaxis_title="Word",
        yaxis_title="Cultural Affinity (African - Latin shift)",
        height=500, width=1100,
        template='plotly_white',
        xaxis=dict(tickangle=45)
    )
    fig.show()
    save_figure(fig, "19_word_cultural_affinity.html")

    # Save affinity data
    affinity_df = pd.DataFrame(affinity_data)
    affinity_df.to_csv(os.path.join(config.output_dir, "word_cultural_affinity.csv"), index=False)
    print(f"💾 Saved: word_cultural_affinity.csv")
else:
    print("⚠️ Need both African and Latin models for affinity analysis")


📊 Generating Word-to-Model Affinity Analysis...


💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/19_word_cultural_affinity.html


💾 Saved: word_cultural_affinity.csv


In [96]:
# ============================================================================
# CELL 44: PLOT 20 - COMPREHENSIVE 3D COMPARISON (ALL MODELS)
# ============================================================================

print("\n📊 Generating Comprehensive 3D Model Comparison...")

from sklearn.decomposition import PCA

# Collect all embeddings
all_embeddings = []
all_labels = []
all_model_names = []

for word in ALL_WORDS:
    all_embeddings.append(base_isolated_words[word][ANALYSIS_LAYERS[-1]]['embedding'])
    all_labels.append(word)
    all_model_names.append('Base')

if african_isolated_words:
    for word in ALL_WORDS:
        all_embeddings.append(african_isolated_words[word][ANALYSIS_LAYERS[-1]]['embedding'])
        all_labels.append(word)
        all_model_names.append('African')

if latin_isolated_words:
    for word in ALL_WORDS:
        all_embeddings.append(latin_isolated_words[word][ANALYSIS_LAYERS[-1]]['embedding'])
        all_labels.append(word)
        all_model_names.append('Latin')

if offspring_isolated_words:
    for word in ALL_WORDS:
        all_embeddings.append(offspring_isolated_words[word][ANALYSIS_LAYERS[-1]]['embedding'])
        all_labels.append(word)
        all_model_names.append('Offspring')

# PCA on combined embeddings
all_embeddings = np.array(all_embeddings)
pca = PCA(n_components=3)
all_3d = pca.fit_transform(all_embeddings)

fig = go.Figure()

unique_models = list(set(all_model_names))
for model_name in unique_models:
    mask = [m == model_name for m in all_model_names]
    indices = [i for i, m in enumerate(mask) if m]

    color = MODEL_COLORS.get(model_name.lower(), 'red')

    fig.add_trace(go.Scatter3d(
        x=all_3d[indices, 0],
        y=all_3d[indices, 1],
        z=all_3d[indices, 2],
        mode='markers+text',
        name=model_name,
        marker=dict(size=6, color=color, opacity=0.8),
        text=[all_labels[i][:4] for i in indices],
        textposition='top center',
        textfont=dict(size=8),
    ))

# Draw lines connecting same word across models
if len(unique_models) > 1:
    for word in ALL_WORDS[:5]:  # Just first 5 to avoid clutter
        word_indices = [i for i, l in enumerate(all_labels) if l == word]
        if len(word_indices) > 1:
            for i in range(len(word_indices) - 1):
                fig.add_trace(go.Scatter3d(
                    x=[all_3d[word_indices[i], 0], all_3d[word_indices[i+1], 0]],
                    y=[all_3d[word_indices[i], 1], all_3d[word_indices[i+1], 1]],
                    z=[all_3d[word_indices[i], 2], all_3d[word_indices[i+1], 2]],
                    mode='lines',
                    line=dict(color='gray', width=1, dash='dot'),
                    showlegend=False,
                    hoverinfo='skip'
                ))

fig.update_layout(
    title=dict(text="3D Word Embedding Space: All Models<br><sup>Same words connected across models</sup>",
               font=dict(size=16)),
    scene=dict(
        xaxis_title=f"PC1 ({pca.explained_variance_ratio_[0]:.1%})",
        yaxis_title=f"PC2 ({pca.explained_variance_ratio_[1]:.1%})",
        zaxis_title=f"PC3 ({pca.explained_variance_ratio_[2]:.1%})",
    ),
    height=700, width=900,
    template='plotly_white'
)
fig.show()
save_figure(fig, "20_all_models_3d_embedding_space.html")


📊 Generating Comprehensive 3D Model Comparison...


💾 Saved: /content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/20_all_models_3d_embedding_space.html


'/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/01Jan2026/ndna_validated_results/20_all_models_3d_embedding_space.html'

In [97]:
# ============================================================================
# CELL 45: FINAL SUMMARY AND VALIDATION REPORT
# ============================================================================

print("\n" + "=" * 70)
print("✅ nDNA CULTURAL MODEL ANALYSIS - FINAL SUMMARY")
print("=" * 70)

# Validation checks
print("\n🔍 VALIDATION CHECKS:")
print("-" * 50)

# Check 1: Opposite words should be dissimilar
war_peace_sim = isolated_analyzer.compute_cosine_similarity(
    base_isolated_words['war'][ANALYSIS_LAYERS[-1]]['embedding'],
    base_isolated_words['peace'][ANALYSIS_LAYERS[-1]]['embedding']
)
print(f"1. war ↔ peace similarity: {war_peace_sim:.4f}")
print(f"   {'✅ PASS' if war_peace_sim < 0.7 else '⚠️ UNEXPECTED'}: Opposite meanings should be dissimilar")

# Check 2: Similar words should be similar
war_destroy_sim = isolated_analyzer.compute_cosine_similarity(
    base_isolated_words['war'][ANALYSIS_LAYERS[-1]]['embedding'],
    base_isolated_words['destroy'][ANALYSIS_LAYERS[-1]]['embedding']
)
print(f"2. war ↔ destroy similarity: {war_destroy_sim:.4f}")
print(f"   {'✅ PASS' if war_destroy_sim > 0.5 else '⚠️ CHECK'}: Similar meanings should be close")

# Check 3: Fine-tuned models should differ from base
if african_isolated_words:
    african_culture_shift = 1 - isolated_analyzer.compute_cosine_similarity(
        base_isolated_words['culture'][ANALYSIS_LAYERS[-1]]['embedding'],
        african_isolated_words['culture'][ANALYSIS_LAYERS[-1]]['embedding']
    )
    print(f"3. African 'culture' shift from base: {african_culture_shift:.4f}")
    print(f"   {'✅ PASS' if african_culture_shift > 0.01 else '⚠️ LOW SHIFT'}: Fine-tuning should change embeddings")

# Check 4: Offspring should be between parents
if offspring_isolated_words and african_isolated_words and latin_isolated_words:
    offspring_african = isolated_analyzer.compute_cosine_similarity(
        offspring_isolated_words['culture'][ANALYSIS_LAYERS[-1]]['embedding'],
        african_isolated_words['culture'][ANALYSIS_LAYERS[-1]]['embedding']
    )
    offspring_latin = isolated_analyzer.compute_cosine_similarity(
        offspring_isolated_words['culture'][ANALYSIS_LAYERS[-1]]['embedding'],
        latin_isolated_words['culture'][ANALYSIS_LAYERS[-1]]['embedding']
    )
    print(f"4. Offspring 'culture' ↔ African: {offspring_african:.4f}")
    print(f"   Offspring 'culture' ↔ Latin: {offspring_latin:.4f}")
    print(f"   {'✅ PASS' if offspring_african > 0.9 and offspring_latin > 0.9 else '⚠️ CHECK'}: Offspring should be similar to both parents")

print("\n📁 OUTPUT FILES GENERATED:")
print("-" * 50)
for f in sorted(os.listdir(config.output_dir)):
    filepath = os.path.join(config.output_dir, f)
    size = os.path.getsize(filepath)
    print(f"   • {f} ({size/1024:.1f} KB)")

print("\n" + "=" * 70)
print("🎉 ANALYSIS COMPLETE!")
print("=" * 70)


✅ nDNA CULTURAL MODEL ANALYSIS - FINAL SUMMARY

🔍 VALIDATION CHECKS:
--------------------------------------------------
1. war ↔ peace similarity: 0.4662
   ✅ PASS: Opposite meanings should be dissimilar
2. war ↔ destroy similarity: 0.4449
   ⚠️ CHECK: Similar meanings should be close
3. African 'culture' shift from base: 0.4677
   ✅ PASS: Fine-tuning should change embeddings
4. Offspring 'culture' ↔ African: 0.9227
   Offspring 'culture' ↔ Latin: 0.9245
   ✅ PASS: Offspring should be similar to both parents

📁 OUTPUT FILES GENERATED:
--------------------------------------------------
   • 01_all_models_ndna_comparison.html (4470.9 KB)
   • 02_3d_ndna_trajectory.html (4465.3 KB)
   • 03_word_similarity_heatmaps_base.html (4510.6 KB)
   • 04_word_similarity_all_models_last_layer.html (4550.2 KB)
   • 05_word_embedding_norm_base.html (4482.3 KB)
   • 06_category_similarity_analysis.html (4460.2 KB)
   • 07_key_word_pairs_all_models.html (4461.0 KB)
   • 08_word_pair_evolution_layers.htm